# 1. Data reading / writing

In [ ]:
import scanpy as sc
import pandas as pd

In [ ]:
### visium hd
#optional:
'''
pathout = '/data/kanferg/Sptial_Omics/projects/NatalieLab/liver_cancer/spatialomicstoolkit/out_1'
path_in = '/data/kanferg/Sptial_Omics/projects/NatalieLab/liver_cancer/data/CS037675/SCAF4363_24008097_A1_VHD/PrimaryAnalysisOutput/SCAF4363_24008097_A1_PA_VHD/outs/binned_outputs/square_016um'
'''
def parquet_to_csv(path):
    '''
    Converts a Parquet file to a CSV file if the CSV file does not already exist.
    '''
    file_path = os.path.join(path,'spatial/tissue_positions_list.csv')
    if not os.path.exists(file_path):
        df = pd.read_parquet(os.path.join(path,'spatial/tissue_positions.parquet'))
        # Write to a CSV file
        df.to_csv(os.path.join(path,'spatial/tissue_positions_list.csv'), index=False)
    return
parquet_to_csv(path_in)
andata = sc.read_visium(path=path_in)

# xenium
# path = '/data/HiTIF/data/spatialomics/melanoma/data/toxicology/SCAF4481/PrimaryAnalysisOutput/SCAF4481_PA_xenium/output-XETG00202__0059711_Right__SCAF04481_Right__R1__20250403__144627'
sdata = spatialdata_io.xenium(
        path=str(path),
        n_jobs=8,
        cells_boundaries=True,
        nucleus_boundaries=False,
        morphology_focus=True,
        cells_as_circles=False,
    )
path = '/data/HiTIF/data/spatialomics/melanoma/data/toxicology/SCAF4481/PrimaryAnalysisOutput/SCAF4481_PA_xenium/output-XETG00202__0059711_Right__SCAF04481_Right__R1__20250403__144627'
path_zarr = path + '.zarr'
sdata.write(path_zarr,overwrite=True)

path = '/data/HiTIF/data/spatialomics/melanoma/data/toxicology/SCAF4481/PrimaryAnalysisOutput/SCAF4481_PA_xenium/output-XETG00202__0059711_Right__SCAF04481_Right__R1__20250403__144627'
path_zarr = path + '.zarr'
sdata = sd.read_zarr(path_zarr)

# building andata from multiple andata
#example 1:
'''
path2name = {'output-XETG00202__0052749_Left__SCAF04536_Left_R1__20250508__163439':'SCAF04536_Left_R1',
            'output-XETG00202__0052749_Left__SCAF04536_Left_R2__20250508__163439':'SCAF04536_Left_R2',
            'output-XETG00202__0052749_Left__SCAF04536_Left_R3__20250508__163439':'SCAF04536_Left_R3',
            'output-XETG00202__0052764_Right__SCAF04537_Right_R1__20250508__163439':'SCAF04537_Right_R1',
             'output-XETG00202__0052764_Right__SCAF04537_Right_R2__20250508__163439':'SCAF04537_Right_R2',
             'output-XETG00202__0052764_Right__SCAF04537_Right_R3__20250508__163439':'SCAF04537_Right_R3'}
path_left = '/data/kanferg/Sptial_Omics/projects/StrackerLab/brain_inflammation/data/SCAF4536/PrimaryAnalysisOutput/SCAF4536_PA_xenium'
path_right = '/data/kanferg/Sptial_Omics/projects/StrackerLab/brain_inflammation/data/SCAF4537/PrimaryAnalysisOutput/SCAF4537_PA_xenium'
outPath = '/data/kanferg/Sptial_Omics/projects/StrackerLab/brain_inflammation/spatialomicstoolkit/out'
'''
Batch_list = []
batch_uneique = []
for key in path2name.keys():
    if 'Left' in key:
        andata_temp = StDatareader_rsc(path = os.path.join(path_left,key), outPath = outPath , FilePrefix = "",hdffileName = "",method = "xenium")
    else:
        andata_temp = StDatareader_rsc(path = os.path.join(path_right,key), outPath = outPath , FilePrefix = "",hdffileName = "",method = "xenium")
    andata = andata_temp.andata
    andata.obs['sample'] = path2name[key]
    batch_uneique.append(path2name[key])
    Batch_list.append(andata)
for a in Batch_list:
    a.var_names_make_unique()          # avoid duplicate gene names
    # If you have categorical columns, make them plain strings so merge works
    for col in a.var.columns:
        if a.var[col].dtype.name == "category":
            a.var[col] = a.var[col].astype(str)
            
import warnings
warnings.filterwarnings(
    "ignore",
    message="Ignoring `palette` because no `hue` variable has been assigned."
)

# example 2:
'''
PATH_ANDATA  = "/data/HiTIF/data/spatialomics/cystic_duct/andata_files"
PATH_META    = "/data/kanferg/Sptial_Omics/projects/HernandezLab/cystic_duct/data/2026_4_22_Annotations_for_Gil.csv"
'''
# --- metadata ----------------------------------------------------------------
meta = pd.read_csv(PATH_META)
meta.columns = [c.strip() for c in meta.columns]
# prefix used for matching = portion before the first "__"
meta["sample_prefix"] = meta["Sample Name"].str.split("__").str[0]
# tidy
for c in ["Treatment", "Concentration", "Patient", "Sample origin"]:
    meta[c] = meta[c].astype(str).str.strip()
meta["Concentration"] = meta["Concentration"].str.replace(r"\s+", " ", regex=True)
meta.head()

map_p = {'1': "P1", '2': "P2", '3': "P3"}
meta['Patient'] = meta['Patient'].astype(str).map(map_p)
meta['section#'] = "R" + "_" + meta['section#'].astype(str)

meta['sample_key'] = meta['Block'].astype(str) + "_" + meta['Patient'].astype(str) + "_" + meta['section#'] + "_" + meta['Treatment'].astype(str)

# --- load every .h5ad and attach matching metadata --------------------------
files = sorted(glob.glob(os.path.join(PATH_ANDATA, "*.h5ad")))
print(f"Found {len(files)} h5ad files.")

meta_by_prefix = meta.set_index("sample_prefix")
META_COLS = ["Sample Name", "Sample origin", "Sample Prep", "Culture condition",
             "Hours in system", "Patient", "Accession number", "Block",
             "section#", "Treatment", "Concentration", "sample_key"]

# Each Xenium .h5ad may contain multiple rows per physical cell; keep one row
# per cell using the cell_id column.
def _one_row_per_cell(a):
    if "cell_id" in a.obs.columns:
        a = a[~a.obs["cell_id"].duplicated()].copy()
    if "segmentation_method" in a.obs.columns:
        a.obs.drop(columns=["segmentation_method"], inplace=True)
    return a

adatas = {}
for fp in files:
    sample = Path(fp).stem
    a = _one_row_per_cell(sc.read_h5ad(fp))
    # Preserve original obs_names so source cell IDs are recoverable later.
    a.obs["orig_obs_name"] = a.obs_names.astype(str).values
    a.obs_names_make_unique()
    a.var_names_make_unique()
    # NOTE: do NOT set a.obs["sample"] here — ad.concat(..., label="sample",
    # keys=...) will populate it during the merge in section 3.
    if sample in meta_by_prefix.index:
        for c in META_COLS:
            a.obs[c] = meta_by_prefix.loc[sample, c]
    else:
        print(f"  ! no metadata match for {sample}")
    adatas[sample] = a

print("Loaded:", list(adatas.keys()))
print({s: a.n_obs for s, a in adatas.items()})

# Concatenate every sample into one AnnData
adata = ad.concat(
    list(adatas.values()), axis=0, join="inner",
    label="sample", keys=list(adatas.keys()), index_unique="-",
)
adata.obs["sample"] = adata.obs["sample"].astype(str)
adata.layers["counts"] = adata.X.copy()

print(f"Merged: {adata.n_obs} cells x {adata.n_vars} genes  "
      f"({adata.obs['sample'].nunique()} samples)")
print(adata.obs["sample"].value_counts())


# QC

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import ListedColormap,Normalize
from matplotlib.cm import ScalarMappable
import matplotlib.ticker as ticker
import seaborn as sns
import os
import gzip
import numpy as np
import rapids_singlecell as rsc
import pandas as pd
import random
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from scipy.stats import gaussian_kde
from scipy.interpolate import griddata


def set_image_para():
    plt.rcParams['figure.dpi'] = 150
    plt.rcParams['font.family'] = ['serif']
    plt.rcParams['font.size'] = 12
    plt.rcParams['axes.labelsize'] = 12
    plt.rcParams['axes.titlesize'] = 12
    plt.rcParams['xtick.labelsize'] = 12
    plt.rcParams['ytick.labelsize'] = 12


def plot_dist(andata,column,ax,type = 'obs', bins = 'auto',title = '',xlab = '',ylab =''):
    '''
    You can replace 'auto' with any other method (e.g., 'fd', 'doane', 'scott', 'rice', 'sturges', or 'sqrt')
    '''
    palette1 = sns.color_palette("colorblind",9)
    if type == 'obs':
        arr = andata.obs[column].values
    else:
        arr = andata.var[column].values
    bin_edges = np.histogram_bin_edges(arr, bins='auto')
    # Calculate bin edges using NumPy's 'auto' method
    # Calculate bin width
    bin_width = bin_edges[1] - bin_edges[0]
    set_image_para()
    sns.histplot(arr, binwidth=bin_width,palette=palette1,ax = ax, kde=True)
    ax.set_ylabel(ylab)
    ax.set_xlabel(xlab)
    ax.set_title(title)
    
def custom_paramsForSPatialPlot():
    custom_params = {"xtick.labelsize": 0,      
                    "ytick.labelsize": 0,      
                    "axes.labelsize": 0,       
                    "xtick.major.size": 0,     
                    "xtick.minor.size": 0,     
                    "ytick.major.size": 0,    
                    "ytick.minor.size": 0 }
    return custom_params
    
def plot_bin2d(andata,ax,title = '',xlab = '',ylab =''):
    palette1 = sns.color_palette("colorblind",10)
    ax.scatter(andata.obs['total_counts'],andata.obs['n_genes_by_counts'], alpha=0.6,color = palette1[0],edgecolor='black')
    ax.set_ylabel(ylab)
    ax.set_xlabel(xlab)
    ax.set_title(title)
    
def plot_spatial_data(andata, column, ax,fig, size = 2, set_xlabel_cbar = '', **kwargs):
    df = pd.DataFrame({
        str(column): andata.obs[column],
        'x': andata.obsm['spatial'][:, 0],
        'y': andata.obsm['spatial'][:, 1],
        'total_counts': andata.obs['total_counts']
    })
    
    
    palette = sns.color_palette("Blues", as_cmap=True)
    listed_cmap = ListedColormap(palette(np.linspace(0, 1, 256)))
    
    norm = Normalize(vmin=df[column].min(), vmax=df[column].max())
    sc = ax.scatter(x=df['x'], y=df['y'], c=df[column], cmap=listed_cmap, norm=norm, s=size, **kwargs)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    
    cbar = fig.colorbar(ScalarMappable(norm=norm, cmap=listed_cmap), ax=ax)
    cbar.ax.set_xlabel(set_xlabel_cbar, labelpad=10)
    cbar.ax.xaxis.set_label_position('top')
    cbar.ax.xaxis.label.set_size(10)  # Reduce label font size
    cbar.ax.tick_params(labelsize=8) 
    
    return sc

def plot_spatial(andata,ax, cluster = 'cluster', features = None,title = '',xlab = '',ylab ='',size = 2,alpha = 0.6, markerscale = 5, cluster_name = 'cluster'):
    palette = sns.color_palette("tab20") + sns.color_palette("tab20b") + sns.color_palette("tab20c")
    df = pd.DataFrame({'cluster':andata.obs[cluster],'x':andata.obsm['spatial'][:,0],'y':andata.obsm['spatial'][:,1]})
    if features:
        df[df['cluster'].isin([features])]
    num_classes = len(df['cluster'].unique())
    if num_classes==1:
        listed_cmap = ListedColormap(palette)
    else:
        num_classes = len(np.unique(df['cluster'].values))
        extended_palette = palette * (num_classes // len(palette) + 1)
        extended_palette = extended_palette[:num_classes]
        listed_cmap = ListedColormap(extended_palette)
    color_container = []
    
    clusters = sorted(np.unique(df['cluster'].values), key=int)
    for i, cluster in enumerate(clusters):
        cluster_data = df[df['cluster'] == cluster]
        ax.scatter( x=cluster_data['x'], y=cluster_data['y'], color=listed_cmap(i), label=f'{cluster}', s=size, alpha=alpha)
        color_container.append(listed_cmap(i))
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)
    ax.set_title(title)
    legend = ax.legend( title=cluster_name,
                        bbox_to_anchor=(1.05, 1),  # Position the legend outside the plot
                        loc='upper left',
                        fontsize='small',  # Control the font size
                        title_fontsize='medium',
                        markerscale=markerscale,  # Increase the size of the legend markers
                        frameon=False# Control the title font size
                        )
    df_color = pd.DataFrame({"clusters":df['cluster'].unique(),"colors":color_container})
    return df_color
    
def plot_expression(df,marker,ax,**kwargs):
    custom_params = {"axes.spines.right": False, "axes.spines.top": False}
    sns.set_theme(style="ticks", rc=custom_params)  
    g = sns.violinplot(data = df, x = "cluster", y = marker, ax = ax, **kwargs)
    max_value = df[marker].max()
    g.set(ylim = (0,max_value+1))
    ax.text(0.95, 0.95, marker, transform=ax.transAxes, fontsize=12, verticalalignment='top',horizontalalignment='right')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
    ax.tick_params(axis='x', labelsize=8)
    ax.set_ylabel('read count \n [log-normalized]')
    ax.set_xlabel('')
    
    
def plot_moran(andata,feature, figsize =(5,4),xlabel = '',ylabel = '',title = '', legend_title = '', **kwargs):
    from rsc_functions.utility.SpatialStats import compute_spatial_lag
    compute_spatial_lag(andata = andata,feature = feature)
    lagged_total_counts = andata.obs[f'lagged_{feature}'].values
    data = andata.obs[f'{feature}'].values
    clusters = andata.obs['cluster'].astype('category').cat.codes.values  # Convert clusters to numerical codes
    # Convert to NumPy arrays for plotting and regression
    total_counts_np = np.asarray(data)
    lagged_total_counts_np = np.asarray(lagged_total_counts)
    clusters_np = np.asarray(clusters)

    # Unique clusters
    unique_clusters = np.unique(clusters_np)

    # Prepare the DataFrame for easier handling
    df = pd.DataFrame({
        f'{feature}': total_counts_np,
        f'lagged_{feature}': lagged_total_counts_np,
        'cluster': clusters_np
    })

    palette = sns.color_palette("tab20") + sns.color_palette("tab20b") + sns.color_palette("tab20c")
    num_classes = len(df['cluster'].unique())
    extended_palette = palette * (num_classes // len(palette) + 1)
    extended_palette = extended_palette[:num_classes]
    listed_cmap = ListedColormap(extended_palette)
        
    # custom_params = custom_paramsForSPatialPlot()
    # sns.set_theme(style="whitegrid", palette="pastel", rc=custom_params)
    
    plt.rcParams['figure.dpi'] = 92
    plt.rcParams['font.family'] = ['serif']
    plt.rcParams['font.size'] = 12
    plt.rcParams['axes.labelsize'] = 12
    plt.rcParams['axes.titlesize'] = 12
    plt.rcParams['xtick.labelsize'] = 12
    plt.rcParams['ytick.labelsize'] = 12
    
    r_squared_contain = {}
    fig, ax = plt.subplots(figsize=figsize)
    for i, cluster in enumerate(df['cluster'].unique()):
        # Filter the data for the current cluster
        cluster_data = df[df['cluster'] == cluster]

        # Scatter plot
        ax.scatter(cluster_data[f'{feature}'], cluster_data[f'lagged_{feature}'], color= listed_cmap(i), label=f'{cluster}', edgecolor='black',  **kwargs)

        # Linear regression for the linear fit
        model = LinearRegression()
        model.fit(cluster_data[f'{feature}'].values.reshape(-1, 1), cluster_data[f'lagged_{feature}'].values)
        line_x = np.linspace(cluster_data[f'{feature}'].min(), cluster_data[f'{feature}'].max(), 1000)
        line_y = model.predict(line_x.reshape(-1, 1))

        # Plot the linear fit
        ax.plot(line_x, line_y, color=listed_cmap(i))
        
        # Calculate R-squared value
        r_squared = r2_score(cluster_data['lagged_total_counts'], model.predict(cluster_data['total_counts'].values.reshape(-1, 1)))
        r_squared_contain[f'{cluster}'] = [np.round(r_squared,2)]
        # Plot customization
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    legend = ax.legend(title=legend_title,
                    bbox_to_anchor=(1.05, 1),  # Position the legend outside the plot
                    loc='upper left',
                    fontsize='small',  # Control the font size
                    title_fontsize='medium',
                    markerscale=5,  # Increase the size of the legend markers
                    frameon=False # Control the title font size
                    )
    return r_squared_contain

fig, ax = plt.subplots(6, 4, figsize=(12, 15), dpi=100,constrained_layout=True)
batchs = pd.unique(adata_concat.obs['sample'].astype(str))
for i,b in enumerate(batchs):
    andata_temp = adata_concat[adata_concat.obs['sample']==b].copy()
    plot_dist(andata_temp,column = 'total_counts',ax = ax[i,0],xlab = 'Reads Count\Cells ')
    plot_dist(andata_temp,column = 'n_genes_by_counts',ax = ax[i,1],xlab = 'Genes Count\Cells')
    plot_dist(andata_temp,column = 'log1p_total_counts',bins = 'doane', type = 'obs', ax = ax[i,2],xlab = 'Reads Count\Cells (log)')
    plot_dist(andata_temp,column = 'log1p_n_genes_by_counts',bins = 'doane', type = 'obs', ax = ax[i,3],xlab = 'Genes Count\Cells (log)')
    ax[i, 0].set_ylabel(b, fontsize=10, rotation=90, labelpad=10, weight='bold')



# visiumhd    
sdata_b2c = '/data/kanferg/Sptial_Omics/projects/NatalieLab/liver_cancer/spatialomicstoolkit/out_1/bin2cell/for_spatialdata'
batch = os.listdir(sdata_b2c)
batch.remove('SCAF4333_24008099_A1_VHD_cdata.h5ad')
batch.remove('SCAF4333_24008099_A1_VHD_labels_he.h5ad')


batch_b2c = [b for b in batch if 'cdata' in b]
batch_b2c_dict = {b:i for i,b in enumerate(batch_b2c)}


adata_b2c_c6 = sc.read_h5ad(os.path.join(sdata_b2c,batch_b2c[0]))
adata_b2c_e6 = sc.read_h5ad(os.path.join(sdata_b2c,batch_b2c[1]))
adata_b2c_e4 = sc.read_h5ad(os.path.join(sdata_b2c,batch_b2c[2]))
adata_b2c_c2 = sc.read_h5ad(os.path.join(sdata_b2c,batch_b2c[3]))
adata_b2c_e2 = sc.read_h5ad(os.path.join(sdata_b2c,batch_b2c[4]))


def get_bin(arr):  
    bin_edges = np.histogram_bin_edges(arr, bins='auto')
    # Calculate bin edges using NumPy's 'auto' method
    # Calculate bin width
    bin_width = bin_edges[1] - bin_edges[0]
    return bin_width

def voyger_transform(adata,percentile_sum = 5,percentile_bin = 1):
    adata.var['symbol'] = adata.var.index.values
    is_mt = adata.var['symbol'].str.contains('^mt-').values
    vp.utils.add_per_cell_qcmetrics(adata, subsets={'mito': is_mt})
    plt.style.use('default')
    #plt.rcParams['text.usetex'] = True
    plt.rcParams['font.size'] = 12
    # plt.style.use('fivethirtyeight')
    
    fig, axes = plt.subplots(2, 3,figsize = (14,7))
    axes = axes.ravel()
    axes[0].scatter(adata.obs['detected'].values,adata.obs['subsets_mito_percent'].values,s = 1, alpha=0.6, edgecolor='black')
    axes[1].scatter(adata.obs['sum'].values,adata.obs['detected'].values, alpha=0.6,s = 1, edgecolor='black')
    axes[2].scatter(adata.obs['bin_count'].values,adata.obs['subsets_mito_percent'].values, alpha=0.6,s = 1, edgecolor='black')
    axes[3].scatter(adata.obs['bin_count'].values,adata.obs['sum'].values, alpha=0.6,s = 1, edgecolor='black')
    sns.histplot(adata.obs['sum'].values, binwidth=get_bin(adata.obs['sum'].values), ax = axes[4], kde=True)
    prec_sum =  np.round(np.percentile(adata.obs['sum'].values,percentile_sum),0)
    axes[4].axvline(prec_sum,c = 'red',linestyle='--',label = str(prec_sum))
    axes[4].legend()
    sns.histplot(adata.obs['bin_count'].values, binwidth=get_bin(adata.obs['bin_count'].values), ax = axes[5], kde=True)
    prec_b =  np.round(np.percentile(adata.obs['bin_count'].values,percentile_bin),0)
    axes[5].axvline(prec_b,c = 'red',linestyle='--',label = str(prec_b))
    axes[5].legend()
    
    axes[0].set_xlabel('detected')
    axes[0].set_ylabel('subsets_mito_percent')
    axes[1].set_xlabel('sum')
    axes[1].set_ylabel('subsets_mito_percent')
    axes[2].set_xlabel('bin_count')
    axes[2].set_ylabel('subsets_mito_percent')
    axes[3].set_xlabel('bin_count')
    axes[3].set_ylabel('sum')
    axes[4].set_xlabel('sum')
    axes[5].set_xlabel('bin_count')
    plt.subplots_adjust(wspace = 0.8,hspace = 0.3)
    
class Voyagerpy_processing:
    def __init__(self,adata,bins='auto',percentile_sum = 5,percentile_bin = 1,sum_th = None,mito_th = None,bin_th = None,marker_list = None):
        self.adata = adata
        self.bins = bins
        self.percentile_sum = percentile_sum
        self.percentile_bin = percentile_bin
        self.sum_th = sum_th
        self.mito_th = mito_th
        self.bin_th = bin_th
        self.marker_list = marker_list
        
    def get_bin(self,arr):  
        bin_edges = np.histogram_bin_edges(arr, bins=self.bins)
        # Calculate bin edges using NumPy's 'auto' method
        # Calculate bin width
        bin_width = bin_edges[1] - bin_edges[0]
        return bin_width
    
    def voyger_transform(self):
        self.adata.var['symbol'] = self.adata.var.index.values
        is_mt = self.adata.var['symbol'].str.contains('^mt-').values
        vp.utils.add_per_cell_qcmetrics(self.adata, subsets={'mito': is_mt})
        plt.style.use('default')
        #plt.rcParams['text.usetex'] = True
        plt.rcParams['font.size'] = 12
        # plt.style.use('fivethirtyeight')
        
        fig, axes = plt.subplots(2, 3,figsize = (14,7))
        axes = axes.ravel()
        axes[0].scatter(self.adata.obs['detected'].values,self.adata.obs['subsets_mito_percent'].values,s = 1, alpha=0.6, edgecolor='black')
        axes[1].scatter(self.adata.obs['sum'].values,self.adata.obs['detected'].values, alpha=0.6,s = 1, edgecolor='black')
        axes[2].scatter(self.adata.obs['bin_count'].values,self.adata.obs['subsets_mito_percent'].values, alpha=0.6,s = 1, edgecolor='black')
        axes[3].scatter(self.adata.obs['bin_count'].values,self.adata.obs['sum'].values, alpha=0.6,s = 1, edgecolor='black')
        sns.histplot(self.adata.obs['sum'].values, binwidth=get_bin(self.adata.obs['sum'].values), ax = axes[4], kde=True)
        prec_sum =  np.round(np.percentile(self.adata.obs['sum'].values,self.percentile_sum),0)
        axes[4].axvline(prec_sum,c = 'red',linestyle='--',label = str(prec_sum))
        axes[4].legend()
        sns.histplot(self.adata.obs['bin_count'].values, binwidth=get_bin(self.adata.obs['bin_count'].values), ax = axes[5], kde=True)
        prec_b =  np.round(np.percentile(self.adata.obs['bin_count'].values,self.percentile_bin),0)
        axes[5].axvline(prec_b,c = 'red',linestyle='--',label = str(prec_b))
        axes[5].legend()
        
        axes[0].set_xlabel('detected')
        axes[0].set_ylabel('subsets_mito_percent')
        axes[1].set_xlabel('sum')
        axes[1].set_ylabel('subsets_mito_percent')
        axes[2].set_xlabel('bin_count')
        axes[2].set_ylabel('subsets_mito_percent')
        axes[3].set_xlabel('bin_count')
        axes[3].set_ylabel('sum')
        axes[4].set_xlabel('sum')
        axes[5].set_xlabel('bin_count')
        plt.subplots_adjust(wspace = 0.8,hspace = 0.3)


    def process(self):
        if 'symbol' not in self.adata.var.columns:
            raise ValueError("Column 'symbol' not found in adata.var. Please run voyager_transform before  process.")
        before_process = self.adata.shape[0]
        keep = self.adata.obs.query(f'sum > {self.sum_th} and bin_count > {self.bin_th} and subsets_mito_percent < {self.mito_th}')['object_id'].to_list()
        #self.adata = self.adata[self.adata.obs['object_id'].isin(list(keep))]
        self.adata._inplace_subset_obs(self.adata.obs['object_id'].isin(list(keep)))
        self.adata.layers['counts'] = self.adata.X.copy()
        vp.utils.log_norm_counts(self.adata, inplace=True)
        self.adata.layers['log'] = self.adata.X.copy()
        gene_var = vp.utils.model_gene_var(self.adata, gene_names=self.adata.var_names)
        hvgs = vp.utils.get_top_hvgs(gene_var,n = 10_000)
        
        # Set the 'highly_variable' column for the genes
        self.adata.var['highly_variable'] = False
        self.adata.var.loc[hvgs, 'highly_variable'] = True
        markers = np.intersect1d(self.marker_list,list(self.adata.var.index.values))
        self.adata.var.loc[markers,'highly_variable'] = True
        after_process = self.adata.shape[0]
        print(f"{before_process - after_process} cells removed total number of cells {after_process}")
        

voy = Voyagerpy_processing(adata_b2c_c6)
voy.voyger_transform()

voy.sum_th = 550
voy.mito_th = 12
voy.bin_th = 5
voy.marker_list = marker_genes_list
voy.process()


# more xenium example



"""QC histogram grid — one row per sample, four columns:
  total_counts, n_genes_by_counts, log(total_counts), log(n_genes_by_counts).
Bars are grey, KDE overlay is black.

Sizing and fonts follow Nature publication requirements:
  - Font: Arial, minimum 7 pt labels / 8 pt axis titles
  - Panel width ~45 mm (4 panels = 180 mm ≈ Nature double-column 183 mm)
  - Panel height ~38 mm
  - Line widths ≥ 0.5 pt
  - Export at 300 dpi (colour) / 600 dpi (line art)
  - PDF font type 42 (TrueType outlines)
"""

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ---- Nature-compliant rcParams applied locally per figure -------------------
_NATURE_RC = {
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7,            # base (Nature minimum for figure text)
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}

# Panel dimensions in inches (Nature double-column = 183 mm = 7.2 in)
# 4 panels across → each ~1.77 in (45 mm); height ~1.5 in (38 mm).
_PANEL_W = 45 / 25.4   # mm → in
_PANEL_H = 38 / 25.4


def plot_qc_histograms(
    adata,
    sample_col="sample_key",
    counts_col="total_counts",
    genes_col="n_genes_by_counts",
    bins=60,
    figsize_per_panel=(_PANEL_W, _PANEL_H),
    save_path=None,
    dpi=300,
):
    """Create a multi-row histogram grid (one row per sample) with 4 columns:
    raw counts, raw genes, log(counts), log(genes).

    Parameters
    ----------
    adata : AnnData
        Must have `counts_col` and `genes_col` in `.obs`.
    sample_col : str
        Column used to split rows (default "sample_key").
    counts_col, genes_col : str
        obs columns for total counts and detected genes.
    bins : int
        Number of histogram bins.
    figsize_per_panel : tuple
        (width, height) in inches per individual subplot.
    save_path : str or Path, optional
        If given, saves the figure (PNG + PDF).
    dpi : int
        Resolution for saved PNG (PDF is vector).

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    obs = adata.obs.copy()
    samples = sorted(obs[sample_col].unique())
    n_samples = len(samples)

    ncols = 2
    fw, fh = figsize_per_panel

    with mpl.rc_context(_NATURE_RC):
        fig, axes = plt.subplots(
            n_samples, ncols,
            figsize=(fw * ncols, fh * n_samples),
            constrained_layout=True,
        )
        if n_samples == 1:
            axes = axes[np.newaxis, :]

        col_labels = [
            (counts_col, False, "Reads Count\\Cells"),
            (genes_col,  False, "Genes Count\\Cells"),
            # (counts_col, True,  "Reads Count\\Cells (log)"),
            # (genes_col,  True,  "Genes Count\\Cells (log)"),
        ]

        for row_idx, sample in enumerate(samples):
            sub = obs[obs[sample_col] == sample]
            for col_idx, (col, use_log, xlabel) in enumerate(col_labels):
                ax = axes[row_idx, col_idx]
                vals = sub[col].values.astype(float)
                if use_log:
                    vals = np.log1p(vals)

                sns.histplot(
                    vals, bins=bins, ax=ax,
                    color="#B0B0B0", edgecolor="white", linewidth=0.3,
                    stat="count", kde=True,
                    line_kws={"color": "black", "linewidth": 0.8},
                )
                ax.set_xlabel(xlabel)
                if col_idx == 0:
                    ax.set_ylabel(sample, fontsize=8, fontweight="bold")
                else:
                    ax.set_ylabel("")

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(save_path), dpi=dpi, bbox_inches="tight")
        # Also save a PDF for vector figures (Nature prefers vector)
        pdf_path = save_path.with_suffix(".pdf")
        fig.savefig(str(pdf_path), bbox_inches="tight")

    return fig


# ---------------------------------------------------------------------------
# Box-plot panel: reads/cell, genes/cell, cells/gene
# ---------------------------------------------------------------------------

def plot_qc_boxplots(
    adata,
    sample_col="sample_key",
    counts_col="total_counts",
    genes_col="n_genes_by_counts",
    save_path=None,
    dpi=300,
):
    """Three-panel box plot (one per metric) grouped by sample.

    Box shows 5th, 25th, 50th, 75th, 95th percentiles.
    Black frame, grey fill — same publication style as the histogram grid.

    Metrics:
      1. Reads (UMI) count per cell
      2. Genes detected per cell
      3. Cells per gene (number of cells expressing each gene)

    Parameters
    ----------
    adata : AnnData
    sample_col : str
    counts_col, genes_col : str
    save_path : str or Path, optional
    dpi : int

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    import matplotlib as mpl

    obs = adata.obs.copy()
    samples = sorted(obs[sample_col].unique())

    # Compute cells-per-gene per sample
    from scipy.sparse import issparse
    X = adata.layers.get("counts", adata.X)
    cells_per_gene_data = []
    sample_arr = obs[sample_col].values
    for s in samples:
        mask = (sample_arr == s)
        Xs = X[mask]
        cpg = np.asarray((Xs > 0).sum(axis=0)).ravel()
        cells_per_gene_data.append(pd.DataFrame({
            "value": cpg,
            sample_col: s,
            "metric": "Cells\\Gene",
        }))
    cpg_df = pd.concat(cells_per_gene_data, ignore_index=True)

    # Build long-form dataframes for the other two metrics
    reads_df = obs[[sample_col, counts_col]].rename(columns={counts_col: "value"})
    reads_df["metric"] = "Reads Count\\Cell"
    genes_df = obs[[sample_col, genes_col]].rename(columns={genes_col: "value"})
    genes_df["metric"] = "Genes Count\\Cell"

    metrics_order = ["Reads Count\\Cell", "Genes Count\\Cell", "Cells\\Gene"]
    long = pd.concat([reads_df, genes_df, cpg_df], ignore_index=True)

    # Nature-style sizing: 3 panels across double-column (183 mm)
    panel_w = 60 / 25.4   # 60 mm each → 180 mm total
    panel_h = 55 / 25.4   # 55 mm height

    with mpl.rc_context(_NATURE_RC):
        fig, axes = plt.subplots(
            1, 3, figsize=(panel_w * 3, panel_h),
            constrained_layout=True,
        )
        for ax, metric in zip(axes, metrics_order):
            sub = long[long["metric"] == metric]

            # Custom percentile box: whis = [5, 95]
            bp = ax.boxplot(
                [sub.loc[sub[sample_col] == s, "value"].values for s in samples],
                labels=samples,
                whis=[5, 95],       # whiskers at 5th and 95th percentile
                showfliers=False,
                patch_artist=True,
                medianprops=dict(color="black", linewidth=0.8),
                boxprops=dict(facecolor="#D0D0D0", edgecolor="black", linewidth=0.6),
                whiskerprops=dict(color="black", linewidth=0.6),
                capprops=dict(color="black", linewidth=0.6),
            )
            ax.set_title(metric, fontsize=8, fontweight="bold")
            ax.set_ylabel("")
            ax.tick_params(axis="x", rotation=90)
            ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=10))
            ax.yaxis.set_minor_locator(mpl.ticker.AutoMinorLocator(2))

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(save_path), dpi=dpi, bbox_inches="tight")
        fig.savefig(str(save_path.with_suffix(".pdf")), bbox_inches="tight")

    return fig

sc.pp.calculate_qc_metrics(adata, layer="counts", percent_top=None,
                           log1p=False, inplace=True)
qc_long = adata.obs.copy()
qc_long.head()

order = sorted(qc_long["sample_key"].unique())
counts_df = qc_long.groupby(['sample_key', 'sample']).size().reset_index(name='entry_count')

# Ensure the plot follows your 'order'
counts_df['sample_key'] = pd.Categorical(counts_df['sample_key'], categories=order, ordered=True)
counts_df = counts_df.sort_values('sample_key')

fig, ax = plt.subplots(figsize=(max(4, 0.4*len(order)), 3.2))

# Use hue to differentiate samples and enable legend data
sns.barplot(
    data=counts_df, 
    x='sample_key', 
    y='entry_count', 
    hue='sample', 
    dodge=False, 
    ax=ax, 
    palette=["#0B0F15"] # Keeps all bars the same color as requested
)

ax.set_ylabel("Cells")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=90)
ax.set_title("Cells per sample")

# Place legend outside of the box
ax.legend(
    title="Samples", 
    bbox_to_anchor=(1.05, 1), 
    loc='upper left', 
    fontsize='x-small', 
    title_fontsize='small',
    frameon=False
)

fig.tight_layout()
plt.show()


import sys
sys.path.insert(0, ".")
from helper.qc_plots import plot_qc_histograms

fig = plot_qc_histograms(
    adata[adata.obs.sample_key.str.startswith('C1')],
    sample_col="sample_key",
    counts_col="total_counts",
    genes_col="n_genes_by_counts",
    bins=60,
    save_path=None,
)
plt.show()

from helper.qc_plots import plot_qc_boxplots

fig = plot_qc_boxplots(
    adata,
    sample_col="sample_key",
    counts_col="total_counts",
    genes_col="n_genes_by_counts",
    save_path=None,
)
plt.show()

def spatial_grid(adata, color="total_counts", name="spatial_total_counts",
                 ncols=4, vmax_pct=99, point_size=1.2, cmap="viridis"):
    """Multi-panel spatial map of `color` per sample."""
    samples = sorted(adata.obs["sample_key"].unique())
    nrows = int(np.ceil(len(samples) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.4*ncols, 3.4*nrows),
                             constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    vmax = np.percentile(adata.obs[color], vmax_pct)
    for ax, s in zip(axes, samples):
        sub = adata[adata.obs["sample_key"] == s]
        xy = sub.obsm["spatial"]
        sc_ = ax.scatter(xy[:, 0], -xy[:, 1], c=sub.obs[color],
                         s=point_size, cmap=cmap, vmin=0, vmax=vmax,
                         linewidths=0, rasterized=True)
        ax.set_title(s, fontsize=9)
        ax.set_aspect("equal")
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)
    for ax in axes[len(samples):]:
        ax.axis("off")
    cbar = fig.colorbar(sc_, ax=axes.tolist(), shrink=0.6, pad=0.01)
    cbar.set_label(color)
    fig.suptitle(f"Spatial {color} per sample", fontsize=11)
    savefig(fig, name)
    return fig

spatial_grid(adata, color="total_counts",      name="spatial_total_counts")
spatial_grid(adata, color="n_genes_by_counts", name="spatial_n_genes")

def sample_summary(adata, sample_col, counts_col="total_counts",
                   genes_col="n_genes_by_counts", area_col="cell_area"):
    """Expanded per-sample QC table for filtering decisions."""
    obs = adata.obs
    n_genes_in_panel = n_genes_in_panel = adata.n_vars
    

    records = []
    for s, sub in obs.groupby(sample_col):
        tc = sub[counts_col].values
        ng = sub[genes_col].values
        area = sub[area_col].values if area_col in sub.columns else np.full(len(sub), np.nan)
        # Compute cells per gene for this sample
        X_sub = adata[sub.index].layers["counts"]
        
        cells_per_gene = np.asarray((X_sub > 0).sum(axis=0)).ravel()
        
        #cells_per_gene = (X_sub > 0).sum(axis=0).ravel()

        records.append({
            "sample_key": s,
            "n_cells": len(sub),
            # --- total counts per cell ---
            "counts_mean": np.mean(tc),
            "counts_median": np.median(tc),
            "counts_p5": np.percentile(tc, 5),
            "counts_p25": np.percentile(tc, 25),
            "counts_p75": np.percentile(tc, 75),
            "counts_p95": np.percentile(tc, 95),
            # --- genes per cell ---
            "genes_mean": np.mean(ng),
            "genes_median": np.median(ng),
            "genes_p5": np.percentile(ng, 5),
            "genes_p25": np.percentile(ng, 25),
            "genes_p75": np.percentile(ng, 75),
            "genes_p95": np.percentile(ng, 95),
            # --- % cells lost at candidate filter thresholds ---
            "pct_counts<10": 100 * np.mean(tc < 10),
            "pct_counts<20": 100 * np.mean(tc < 20),
            "pct_counts<50": 100 * np.mean(tc < 50),
            "pct_genes<5": 100 * np.mean(ng < 5),
            "pct_genes<10": 100 * np.mean(ng < 10),
            "pct_genes<20": 100 * np.mean(ng < 20),
            # --- genes-level: cells per gene ---
            "n_genes_total": n_genes_in_panel,
            "genes_in>=5_cells": int(np.sum(cells_per_gene >= 5)),
            "genes_in>=10_cells": int(np.sum(cells_per_gene >= 10)),
            "genes_in>=20_cells": int(np.sum(cells_per_gene >= 20)),
            "mean_cells_per_gene": np.mean(cells_per_gene),
            "median_cells_per_gene": np.median(cells_per_gene),
            # --- area ---
            "mean_area": np.nanmean(area),
            "median_area": np.nanmedian(area),
        })

    df = pd.DataFrame(records).set_index("sample_key").sort_index()
    return df

summary_df = sample_summary(adata, sample_col="sample_key")
summary_df.round(2)

# Preprocessing / normalization

In [ ]:
# xenium data tissue-wise subset
adata_concat_eye = adata_concat[adata_concat.obs['tissue'].isin(['Eye'])].copy()
adata_concat_heart = adata_concat[adata_concat.obs['tissue'].isin(['Heart'])].copy()
adata_concat_kidney = adata_concat[adata_concat.obs['tissue'].isin(['Kidney'])].copy()
adata_concat_liver = adata_concat[adata_concat.obs['tissue'].isin(['Liver'])].copy()
adata_concat_lung = adata_concat[adata_concat.obs['tissue'].isin(['Lung'])].copy()
adata_concat_spleen = adata_concat[adata_concat.obs['tissue'].isin(['Spleen'])].copy()
adata_concat_tumor = adata_concat[adata_concat.obs['tissue'].isin(['Tumor'])].copy()

def run_modularity(adata,neighbors_key,n_neig,n_pcs = 40, r = 1):
    rsc.pp.neighbors(adata, n_pcs=n_pcs, use_rep='X_pca', n_neighbors=n_neig, key_added = neighbors_key)
    rsc.tl.leiden(adata, resolution = 1, random_state=1337, key_added=f'cluster_{neighbors_key}', neighbors_key=neighbors_key)
    rsc.tl.umap(adata, neighbors_key=neighbors_key,min_dist=0.3, spread=1.0, random_state=1948)
    adata.obsm[f'X_umap_{neighbors_key}'] = adata.obsm['X_umap']
    del adata.obsm['X_umap']  

def adata_gener(adata_concat,pca = True,n_comps = 50,test_neig = True, n = 0,r = 1, title = 'Eye',neig = None):
    rsc.get.anndata_to_GPU(adata_concat)
    rsc.pp.scale(adata_concat, max_value=10)
    rsc.pp.pca(adata_concat, n_comps=50,random_state=1337, use_highly_variable=False,)
    if pca:
        with plt.rc_context({"figure.figsize": (2, 2.5),   # width, height in inches
                     "figure.dpi": 100}):        # resolution
            sc.pl.pca_variance_ratio(adata_concat,
                                     n_pcs=50,
                                     log=True)
        return 
    if test_neig:
        neig_list = ([60,70,80,90,100,110],[10,20,30,40,50,60])
        from tqdm import tqdm
        neighbors_dict = {f"{i}neig":i for i in neig_list[n]}
        for key in tqdm(list(neighbors_dict.keys())):
            run_modularity(adata = adata_concat,neighbors_key = key,n_neig = neighbors_dict[key],r = r)
        import random 
        random.seed(30)
        palette = sns.color_palette("tab10") + sns.color_palette("tab20b") + sns.color_palette("Set2")
        random.shuffle(palette)
        neighbors_dict = {f"cluster_{i}neig":f"KNN: {i}" for i in neig_list[n]}
        neighbors_dict_basis = {f"X_umap_{i}neig":i for i in neig_list[n]}
        titles  = list(neighbors_dict.values())          # 6 titles
        colors  = list(neighbors_dict.keys())            # 6 obs columns with cluster labels
        bases   = list(neighbors_dict_basis.keys())      # 6 embeddings in .obsm
        fig, axes = plt.subplots(2, 3, figsize=(11, 7), constrained_layout=True)
        axes = axes.ravel()
        
        for ax, basis, color_key, ttl in zip(axes, bases, colors, titles):
            # how many clusters are in this column?
            n_clust = adata_concat.obs[color_key].nunique()
            X = adata_concat.obsm[f'{basis}'] 
            sc.pl.embedding(
                adata_concat,
                basis=basis,                 # e.g. 'umap_n15' → looks for adata_concat.obsm['X_umap_n15']
                color=color_key,             # the column holding cluster labels
                ax=ax,
                s = 5,
                frameon=False,
                show=False,                  # keep drawing deferred until after the loop
                title=f'{ttl}\n({n_clust} clusters)',  # include the count in the title
                palette=palette,
                legend_loc=None              # **hides the legend**
            )
            ax.set_xlim(X[:, 0].min(), X[:, 0].max())
            ax.set_ylim(X[:, 1].min(), X[:, 1].max())
            ax.set_aspect('equal', adjustable='box')
           
        plt.suptitle(title)
        plt.show()
        return
    else:
        def run_modularity_(adata,neighbors_key,n_neig,n_pcs = 40, r = 1):
            rsc.pp.neighbors(adata, n_pcs=n_pcs, use_rep='X_pca', n_neighbors=n_neig, key_added = neighbors_key)
            rsc.tl.leiden(adata, resolution = 1, random_state=1337, key_added=f'cluster_{neighbors_key}', neighbors_key=neighbors_key)
            rsc.tl.umap(adata, neighbors_key=neighbors_key,min_dist=0.3, spread=1.0, random_state=1948)
            adata.obsm[f'X_umap_{neighbors_key}'] = adata.obsm['X_umap']
            del adata.obsm['X_umap']
            return adata
        key = f"{neig}neig"
        adata = run_modularity_(adata = adata_concat,neighbors_key = key,n_neig = neig,r = r)
        return adata
    
# xenium data brain data
adata_concat.layers['count'] = adata_concat.X.copy()
rsc.pp.log1p(adata_concat)
adata_concat.layers['log'] = adata_concat.X.copy()
rsc.pp.scale(adata_concat, max_value=10)

with plt.rc_context({"figure.figsize": (2, 2.5),   # width, height in inches
                     "figure.dpi": 100}):        # resolution
    sc.pl.pca_variance_ratio(adata_concat,
                             n_pcs=100,
                             log=True)
rsc.pp.neighbors(adata_concat, n_pcs=30, use_rep='X_pca', n_neighbors=30, key_added = '30neig')
rsc.tl.leiden(adata_concat, random_state=1337, key_added='cluster_30neig', neighbors_key='30neig')
rsc.tl.umap(adata_concat, neighbors_key="30neig")

def run_modularity(adata,neighbors_key,n_neig):
    rsc.pp.neighbors(adata, n_pcs=10, use_rep='X_pca', n_neighbors=n_neig, key_added = neighbors_key)
    rsc.tl.leiden(adata, random_state=1337, key_added=f'cluster_{neighbors_key}', neighbors_key=neighbors_key)
    rsc.tl.umap(adata, neighbors_key=neighbors_key)
    adata.obsm[f'X_umap_{neighbors_key}'] = adata.obsm['X_umap']
    del adata.obsm['X_umap']
neighbors_dict = {f"{i}neig":i for i in [10,20,30,40,50,60]}  
for key in list(neighbors_dict.keys()):
    run_modularity(adata = adata_concat,neighbors_key = key,n_neig = neighbors_dict[key])
neighbors_dict = {f"cluster_{i}neig":f"KNN: {i}" for i in [10,20,30,40,50,60]}
neighbors_dict_basis = {f"X_umap_{i}neig":i for i in [10,20,30,40,50,60]} 
titles  = list(neighbors_dict.values())          # 6 titles
colors  = list(neighbors_dict.keys())            # 6 obs columns with cluster labels
bases   = list(neighbors_dict_basis.keys())      # 6 embeddings in .obsm
fig, axes = plt.subplots(2, 3, figsize=(11, 7), constrained_layout=True)
axes = axes.ravel()

for ax, basis, color_key, ttl in zip(axes, bases, colors, titles):
    # how many clusters are in this column?
    n_clust = adata_concat.obs[color_key].nunique()
    X = adata_concat.obsm[f'{basis}'] 
    sc.pl.embedding(
        adata_concat,
        basis=basis,                 # e.g. 'umap_n15' → looks for adata_concat.obsm['X_umap_n15']
        color=color_key,             # the column holding cluster labels
        ax=ax,
        s = 5,
        frameon=False,
        show=False,                  # keep drawing deferred until after the loop
        title=f'{ttl}\n({n_clust} clusters)',  # include the count in the title
        palette=palette,
        legend_loc=None              # **hides the legend**
    )
    ax.set_xlim(X[:, 0].min(), X[:, 0].max())
    ax.set_ylim(X[:, 1].min(), X[:, 1].max())
    ax.set_aspect('equal', adjustable='box')     # keep circles round

plt.show()

# select_umap_parameter =  '30neig'
# adata_concat.obsm['X_umap'] = adata_concat.obsm[f'X_umap_{select_umap_parameter}']
# # ---------- 1  flag extreme points ----------
# coords = adata_concat.obsm[f'X_umap_{select_umap_parameter}']      # shape = (n_cells, 2)

# # keep the central 99 % along each axis (adjust as needed)
# mask = (
#     (coords[:, 0] > np.percentile(coords[:, 0], 0.1)) &
#     (coords[:, 0] < np.percentile(coords[:, 0], 99.8)) &
#     (coords[:, 1] > np.percentile(coords[:, 1], 0.1)) &
#     (coords[:, 1] < np.percentile(coords[:, 1], 99.8))
# )

# # ---------- 2  make a trimmed AnnData view ----------
# adata_trim = adata_concat[mask].copy()

# # ---------- 3  plot without the outliers ----------
# titles = ["", "Sample ID"]
# sc.pl.umap(
#     adata_trim,
#     color=[f'cluster_{select_umap_parameter}', 'sample'],
#     legend_loc=None,
#     ncols=2,
#     size=1,         # (Scanpy ≥1.9: 'size'; older versions: 's')
#     wspace=0.1,
#     frameon=False,
#     title=titles
# )

select_umap_parameter =  '20neig'
adata_concat.obsm['X_umap'] = adata_concat.obsm[f'X_umap_{select_umap_parameter}']
titles = ["KNN: 20 \n PCA: 10", "Sample ID"]
sc.pl.umap(
    adata_concat,
    color=[f'cluster_{select_umap_parameter}', 'sample'],
    legend_loc=None,
    ncols=2,
    size=0.5,         # (Scanpy ≥1.9: 'size'; older versions: 's')
    wspace=0.1,
    frameon=False,
    title=titles
)
# andata_save = adata_concat.copy()
# andata_save.write_h5ad(os.path.join(outPath, "andata.h5ad"))

def run_modularity_res(adata, res = 1):
    rsc.tl.leiden(adata, random_state=1337, key_added=f'cluster_20k_resolution_{res}', neighbors_key='20neig',resolution= res)
    
    resolutions = [0.2,0.4,0.6,0.8,1.0]
for res in resolutions:
    run_modularity_res(andata,res = res)
    
# mamba activate scib-metrics
import igraph as ig
import scanpy as sc
import igraph as ig
from scib.metrics import isolated_labels_asw, graph_connectivity
import pandas as pd
import os
from tqdm import tqdm 

outPath = '/data/kanferg/Sptial_Omics/projects/StrackerLab/brain_inflammation/spatialomicstoolkit/out'
andata = sc.read_h5ad(os.path.join(outPath, "andata.h5ad"))
andata.obsp["connectivities"] = andata.obsp["20neig_connectivities"]
andata.obsp["distances"]      = andata.obsp["20neig_distances"]
andata.uns["neighbors"]       = andata.uns["20neig"] 
# adata = andata[andata.obs['sample']=='SCAF04536_Left_R1'].copy()
adata = sc.pp.subsample(andata, fraction = 0.1, random_state=1980,copy = True)

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, message="pandas.value_counts")
warnings.filterwarnings("ignore", category=FutureWarning, message="The default of observed=False")
warnings.filterwarnings("ignore", category=UserWarning, message="iso_threshold is equal to number of batches")

# ──────────────────────────────────────────────────────────────────────────────
# 2. Build an igraph object once from the connectivities matrix
# ──────────────────────────────────────────────────────────────────────────────
g = sc._utils.get_igraph_from_adjacency(
        adata.obsp["20neig_connectivities"], directed=False
)

# ──────────────────────────────────────────────────────────────────────────────
# 3. Score each clustering
# ──────────────────────────────────────────────────────────────────────────────
resolutions = [0.2,0.4,0.6,0.8,1.0]
scores = []
for res in tqdm(resolutions):
    key = f'cluster_20k_resolution_{res}'
    labels = adata.obs[key].astype(int).tolist()
    n_clusters  = adata.obs[key].nunique()
    
    # (a) Modularity
    Q = g.modularity(labels, weights=g.es["weight"])

    #     # (b) Isolated-labels ASW (graph geodesic distances under the hood)
    #     asw = isolated_labels_asw(
    #         adata,
    #         label_key=key,
    #         batch_key="psedu_batch",     # ← change if your batch column has another name;
    #                                #   if no batches, create adata.obs["batch"] = "1"
    #         embed="X_pca",         # ← any embedding present in adata.obsm
    #         scale=True,
    #     )

    # (c) Graph connectivity
    gc = graph_connectivity(
        adata,
        label_key=key
    )

    scores.append(dict(resolution=res, modularity=Q,
                        graph_conn=gc, n_clusters=n_clusters))
df = pd.DataFrame(scores).set_index("resolution")

_ = df.plot(marker="o", y=["modularity", "graph_conn"], secondary_y="n_clusters")

# clustering and evaluate clusters (UMAP)

In [ ]:
# xenium brain
import scanpy as sc
import spatialleiden as sl
import squidpy as sq
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import random
import pandas as pd
import argparse

outPath = '/data/kanferg/Sptial_Omics/projects/StrackerLab/brain_inflammation/spatialomicstoolkit/out'
andata = sc.read_h5ad(os.path.join(outPath, "andata.h5ad"))

from scipy.spatial.distance import pdist, squareform
adata = andata[andata.obs['sample']=='SCAF04536_Left_R1'].copy()
# Extract spatial coordinates
spatial_coords = adata.obsm["spatial"]

# Compute pairwise Euclidean distances
distances = pdist(spatial_coords, metric="euclidean")
percentile_99 = np.percentile(distances, 1)
percentile_99_int = int(np.round(percentile_99))


# Plot histogram of distances
plt.figure(figsize=(6, 4))
plt.hist(distances, bins=100, color="blue", alpha=0.7)

plt.axvline(percentile_99, color="red", linestyle="--", linewidth=1.5)
plt.text(percentile_99, plt.ylim()[0], f"{percentile_99_int}", 
         ha='right', va='baseline', color="black", fontsize=10)

plt.xlabel("Pairwise Distance")
plt.ylabel("Frequency")
plt.title("Distribution of Pairwise Distances")
plt.show()
del adata
del distances

sq.gr.spatial_neighbors(andata, coord_type="generic", radius= 50)
andata.obsp["spatial_connectivities"] = sl.distance2connectivity(
    andata.obsp["spatial_distances"]
)

sl.spatialleiden(andata, layer_ratio=1.5, directed=(False, True), seed=seed, key_added="spatialleiden_resolution_1")
sl.spatialleiden(andata, resolution=(0.4, 0.4), layer_ratio=1.5, directed=(False, True), seed=seed, key_added="spatialleiden_resolution_0.4")
sl.spatialleiden(andata, resolution=(0.4, 0.4), layer_ratio=1.3, directed=(False, True), seed=seed, key_added="spatialleiden_resolution_0.4_Ratio_1.3")

sl.spatialleiden(andata, resolution=(0.8, 0.8), layer_ratio=1.5, directed=(False, True), seed=seed, key_added="spatialleiden_resolution_0.8")

# andothe spatia lieden optional function:
import cupy as cp
import cupyx
import scanpy as sc
import spatialleiden as sl
import squidpy as sq
import numpy as np
from cupyx.scipy.sparse import csr_matrix
import os
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import random
import pandas as pd
import argparse

def run(infile,outfile):
    pathout = '/data/kanferg/Sptial_Omics/projects/NguyenLab/spatialomicstoolkit/out_1'
    #andata = sc.read_h5ad(os.path.join(pathout, "adata_ctrl_2_logNorm_hvg_unintegrated.h5ad"))
    andata = sc.read_h5ad(os.path.join(pathout, infile))
    andata.obsp['connectivities'] = andata.obsp['nontumor_connectivities']
    andata.obsp['distances'] = andata.obsp['nontumor_distances']
    sq.gr.spatial_neighbors(andata, coord_type="generic")
    pathout_spatlied = "/data/kanferg/Sptial_Omics/projects/NguyenLab/spatialomicstoolkit/out_1"
    seed = 42
    sl.spatialleiden(andata, layer_ratio=1.5, directed=(False, True), seed=seed)
    andata_save = andata.copy()
    #andata_save.write_h5ad(os.path.join(pathout, "adata_ctrl_2_logNorm_hvg_unintegrated.h5ad"))
    andata_save.write_h5ad(os.path.join(pathout_spatlied, outfile))

def main():
    # Create the parser
    parser = argparse.ArgumentParser(description='calling saptiallieden')
    # Add an argument for the option selection
    parser.add_argument('--infile', type=str,required=True)
    parser.add_argument('--outfile', type=str,required=True)
    args = parser.parse_args()

    # Call the run function with the selected option and debug mode
    run(args.infile,args.outfile)

if __name__ == '__main__':
    main()

# umap

adata_concat_b2c.uns['cell_type_16um_hpe_colors'] = [
    '#2166ac',  # Hep1
    '#4393c3',  # Hep2
    '#92c5de',  # Hep3
    '#f4a582',  # Hep4
    '#d6604d',  # Hep5
    '#b2182b',  # Hep6
    '#ff00ff',  # Lesion
    '#808080'   # nan
]
adata_concat_b2c.obs["cell_type"] = adata_concat_b2c.obs["cell_type"].astype("category")
sc.tl.paga(adata_concat_b2c, groups="cell_type", neighbors_key='60neig_scvi')
sc.pl.paga(adata_concat_b2c, plot=False)
sc.tl.umap(adata_concat_b2c, min_dist=0.1, init_pos="paga", neighbors_key='60neig_scvi')

adata_concat_b2c.obs['in_lesions'] = adata_concat_b2c.obs['in_lesions'].astype('category')
adata_concat_b2c.obs['Zone_Layers'] = adata_concat_b2c.obs['Zone_Layers'].astype('category')
adata_concat_b2c.obs['sample'] = adata_concat_b2c.obs['sample'].astype('category')
adata_concat_b2c.obs['cell_type_16um_hpe'] = adata_concat_b2c.obs['cell_type_16um_hpe'].astype('category')

import matplotlib.patheffects as pe
import textwrap
from matplotlib.lines import Line2D


class Plot_umap:
    def __init__(self, adata: object, group_select: str, fix_long_legend: bool = False, wrap_width: int = 34, legend_ncol: int = 1):
        self.adata = adata
        self.group_select = group_select
        self.fix_long_legend = fix_long_legend
        self.wrap_width = wrap_width
        self.legend_ncol = legend_ncol

    def process_andata(self):
        self.adata.obs[self.group_select] = self.adata.obs[self.group_select].astype("category")
        present = list(self.adata.obs[self.group_select].cat.categories)

        # Push 'nan' to the bottom of the legend
        if 'nan' in present:
            present = [c for c in present if c != 'nan'] + ['nan']
            self.adata.obs[self.group_select] = self.adata.obs[self.group_select].cat.reorder_categories(present)

        num_map = {ct: f"{i+1}: {ct}" for i, ct in enumerate(present)}
        self.adata.obs["cell_type_num"] = self.adata.obs[self.group_select].astype(str).map(num_map)
        self.adata.obs["cell_type_num"] = self.adata.obs["cell_type_num"].astype("category")
        self.adata.obs["cell_type_num"] = self.adata.obs["cell_type_num"].cat.set_categories(
            [num_map[ct] for ct in present], ordered=True
        )

        # Propagate original group colors → cell_type_num_colors, respecting the new order
        color_key = f"{self.group_select}_colors"
        if color_key in self.adata.uns:
            orig_cats = list(self.adata.obs[self.group_select].cat.categories)
            orig_colors = list(self.adata.uns[color_key])
            color_map = dict(zip(orig_cats, orig_colors))
            self.adata.uns["cell_type_num_colors"] = [color_map[c] for c in present if c in color_map]

    def plot_umap_celltype_numbered_pub(self):

        plt.rcParams["figure.dpi"] = 300
        plt.rcParams["savefig.dpi"] = 300
        plt.rcParams.update({
            "font.size": 7,
            "axes.labelsize": 7,
            "axes.titlesize": 8,
            "xtick.labelsize": 6,
            "ytick.labelsize": 6,
            "legend.fontsize": 6,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        })

        fig, ax = plt.subplots(figsize=(5.2, 3.0), dpi=300)

        legend_loc = None if self.fix_long_legend else "right margin"

        ax_out = sc.pl.embedding(
            self.adata,
            basis="X_umap",
            color="cell_type_num",
            legend_loc=legend_loc,
            frameon=False,
            size=1,
            edgecolors="black",
            linewidths=0.15,
            show=False,
            ax=ax,
        )

        if self.fix_long_legend:
            cats = list(self.adata.obs["cell_type_num"].cat.categories)

            # Use cell_type_num_colors — guaranteed to match the plotted dots
            if "cell_type_num_colors" in self.adata.uns:
                cols = list(self.adata.uns["cell_type_num_colors"])[:len(cats)]
            else:
                cmap = plt.get_cmap("tab20")
                cols = [cmap(i % 20) for i in range(len(cats))]

            labels = ["\n".join(textwrap.wrap(l, width=self.wrap_width)) for l in cats]

            handles = [
                Line2D([0], [0], marker="o", linestyle="",
                    markerfacecolor=c, markeredgecolor=c, markersize=6)
                for c in cols
            ]

            fig.tight_layout(rect=[0, 0, 0.72, 1])

            fig.legend(
                handles, labels,
                loc="center left",
                bbox_to_anchor=(0.74, 0.5),
                frameon=False,
                ncol=self.legend_ncol,
                handletextpad=0.6,
                labelspacing=0.6,
            )
            for coll in ax_out.collections:
                coll.set_rasterized(True)

        ax_out.set_title("", fontsize=6)

        X = self.adata.obsm["X_umap"]
        df = pd.DataFrame(X, columns=["x", "y"], index=self.adata.obs_names)
        df["lab"] = self.adata.obs["cell_type_num"].astype(str).values
        centers = df.groupby("lab")[["x", "y"]].median()

        label_fs = 5.5
        outline_lw = 1.6

        for lab, (x, y) in centers.iterrows():
            num = lab.split(":")[0].strip()
            t = ax.text(x, y, num, ha="center", va="center",
                        fontsize=label_fs, weight="bold")
            t.set_path_effects([pe.withStroke(linewidth=outline_lw, foreground="white")])

        if not self.fix_long_legend:
            plt.tight_layout()

        plt.show()
        
umpa_object = Plot_umap(adata_concat_b2c,  group_select='cell_type_16um_hpe',fix_long_legend=True)
umpa_object.process_andata()

# gene level umap
genes = [
    "Me1",
    "Cidec",
    "Cidea",
    "Aldh1a1",
    "Aldh3a2",
    "Gstm1",
    "S100a10",
    "Slc1a2",
    "Fasn",
    "Fabp1",
    "Cd36",
    "Maoa",
    "Cbr3",
    "Plin2",
    "Nupr1",
    "Cdkn1a",
    "Cln6",
    "Rrm2",
    "Car2",
]
# missing: 'Cidea', 'Fabp1', 'Cbr3', 'Nupr1'
genes = [g for g in genes if g in adata_concat_b2c.var_names]
df = sc.get.obs_df(adata_concat_b2c, keys=genes)
df_log = np.log1p(df)
df_norm = (df_log - df_log.min()) / (df_log.max() - df_log.min())
for g in genes:
    adata_concat_b2c.obs[f'{g}_norm'] = df_norm[g].values
    
norm_genes = [f'{g}_norm' for g in genes]
plt.rcParams["axes.titlesize"] = 12
axs = sc.pl.umap(
    adata_concat_b2c,
    color=norm_genes,
    frameon=False,
    ncols=3,
    wspace=0.2,
    s=10,
    color_map='viridis',
    title=genes,
    colorbar_loc=None,
    show=False)

cbar_ax = axs[-1]
cbar_ax.clear()
cbar_ax.set_axis_off()

import matplotlib.cm as cm
import matplotlib.colors as mcolors
sm = cm.ScalarMappable(cmap='viridis', norm=mcolors.Normalize(vmin=0, vmax=1))
sm.set_array([])

fig = cbar_ax.get_figure()
cbar = fig.colorbar(sm, ax=cbar_ax, fraction=0.9, pad=0.05, label='log1p norm')
cbar.ax.tick_params(labelsize=12)
cbar.set_label('Min-max normalized log expression', fontsize=12)

plt.show()    

# Integration

In [ ]:
# xenium melanoma
'''
pathout = '/data/kanferg/Sptial_Omics/projects/NguyenLab/spatialomicstoolkit/out_1'
adata_concat = sc.read_h5ad(os.path.join(pathout, "andata_filter_logNorm_hvg_leiden.h5ad"))
'''
meta_data = adata_concat.obs
data_mat = adata_concat.obsm["X_pca"]
import harmonypy as hm
ho = hm.run_harmony(data_mat, meta_data, "batch")

adata_concat.obsm["X_pca_before"] = adata_concat.obsm["X_pca"]
adata_concat.obsm["X_pca_Harmony"] = ho.Z_corr.T

# visuam hd
'''
torch.set_float32_matmul_precision("high")
path_sdata_b2c = '/data/kanferg/Sptial_Omics/projects/NatalieLab/liver_cancer/spatialomicstoolkit/out_1/bin2cell/for_spatialdata'
adata_concat = sc.read_h5ad(os.path.join(path_sdata_b2c,"adata_concat_b2c_voy.h5ad"))
'''
adata_concat.X = adata_concat.layers["counts"].copy()

scvi.model.SCVI.setup_anndata(adata_concat, layer = "counts",batch_key='batch')
model_scvi = scvi.model.SCVI(adata_concat)
model_scvi.train(max_epochs=12)
adata_concat.obsm['X_scVI'] = model_scvi.get_latent_representation()
adata_concat.layers['scvi_normalized'] = model_scvi.get_normalized_expression(library_size = 1e4)

# xenium integration evaluation
'''
pathout = '/data/kanferg/Sptial_Omics/projects/NguyenLab/spatialomicstoolkit/out_1'
adata_concat = sc.read_h5ad(os.path.join(pathout, "andata_filter_logNorm_hvg_leiden_harmony_scvi_con_mouse_cov_umap.h5ad"))
'''
from anndata import AnnData
obsm_before = mock_andata.obsm['X_pca_before']
obsm_harmony = mock_andata.obsm['X_pca_Harmony']
obsm_scvi = mock_andata.obsm['X_scVI']
obsm_cell2loc = mock_andata.obsm['X_cell2loc']
obs_batch = mock_andata.obs['batch'].astype(str).tolist()
obs_celltype = mock_andata.obs['cluster'].astype(str).tolist()
obs_n = mock_andata.obs['total_counts'].values

adata = AnnData(mock_andata.layers['counts'], obsm={"spatial": mock_andata.obsm['spatial'],'Unintegrated':obsm_before,'Harmony':obsm_harmony,'scVI':obsm_scvi,'cell2loc':obsm_cell2loc},obs = {"batch":obs_batch,"cellType":obs_celltype,'total_counts':obs_n})

def faiss_hnsw_nn(X: np.ndarray, k: int):
    """Gpu HNSW nearest neighbor search using faiss.

    See https://github.com/nmslib/hnswlib/blob/master/ALGO_PARAMS.md
    for index param details.
    """
    X = np.ascontiguousarray(X, dtype=np.float32)
    res = faiss.StandardGpuResources()
    M = 32
    index = faiss.IndexHNSWFlat(X.shape[1], M, faiss.METRIC_L2)
    gpu_index = faiss.index_cpu_to_gpu(res, 0, index)
    gpu_index.add(X)
    distances, indices = gpu_index.search(X, k)
    del index
    del gpu_index
    # distances are squared
    return NeighborsResults(indices=indices, distances=np.sqrt(distances))


def faiss_brute_force_nn(X: np.ndarray, k: int):
    """Gpu brute force nearest neighbor search using faiss."""
    X = np.ascontiguousarray(X, dtype=np.float32)
    res = faiss.StandardGpuResources()
    index = faiss.IndexFlatL2(X.shape[1])
    gpu_index = faiss.index_cpu_to_gpu(res, 0, index)
    gpu_index.add(X)
    distances, indices = gpu_index.search(X, k)
    del index
    del gpu_index
    # distances are squared
    return NeighborsResults(indices=indices, distances=np.sqrt(distances))

bm = Benchmarker(
    adata,
    batch_key="batch",
    label_key="cellType",
    embedding_obsm_keys=["Unintegrated","Harmony", "scVI","cell2loc"],
    pre_integrated_embedding_obsm_key="Unintegrated",
    bio_conservation_metrics=BioConservation(isolated_labels=False),
    batch_correction_metrics = BatchCorrection(pcr_comparison = False),
    n_jobs=-1,
)
bm.prepare(neighbor_computer=faiss_brute_force_nn)
bm.benchmark()
bm.plot_results_table()

Differential Expression, GSEA, and Pathway Analysis

In [ ]:
# visumhd
'''
agg_adata_pairs = sc.read_h5ad('/data/HiTIF/data/spatialomics/liver_cancer/data/models/comp_analysis/cell_assign_pair_sample_from_2umbins_zone_report_21.h5ad')
'''
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats
import scipy.sparse as sp
import decoupler as dc
import anndata
import scanpy as sc
import pandas as pd
import numpy as np
import os
from scipy import ndimage as ndi
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib as mpl
from scipy.ndimage import distance_transform_edt, map_coordinates

agg_adata_pairs.obs['sample'] = agg_adata_pairs.obs['sample'].astype(str)
exprement_samples = ['exp_2', 'exp_4', 'exp_6']
adata_dge = agg_adata_pairs[agg_adata_pairs.obs['sample'].isin(exprement_samples)].copy()
adata_dge.obs.rename(columns={'in_lesions':'in_lesion'}, inplace=True)

sample_key = "sample"
lesion_key = "in_lesion"

# 0/1 as strings for contrasts
adata_dge.obs[lesion_key] = adata_dge.obs[lesion_key].astype(int).astype(str)

# group label
groups = adata_dge.obs[[sample_key, lesion_key]].astype(str).agg("__".join, axis=1)

# --- 1) count bins per group ---
group_counts = groups.value_counts().sort_index()
print(group_counts)

# --- 2) choose a balanced K based on smallest group ---
min_bins_per_rep = 200   # adjust (100–500 depending on depth)
K_max = 10

min_group_n = int(group_counts.min())
K = max(1, min(K_max, min_group_n // min_bins_per_rep))

print(f"min group bins = {min_group_n}, using K = {K} pseudo-reps per (sample×lesion)")

# If you want EXACT balance within each sample between lesion and non-lesion:
# K_sample = min over samples of min(n_lesion, n_nonlesion) // min_bins_per_rep
by_sample = (
    adata_dge.obs.assign(_group=groups.values)
    .groupby([sample_key, lesion_key])
    .size()
    .unstack(fill_value=0)
)
# compute per-sample feasible K (balanced between lesion/nonlesion)
K_per_sample = (by_sample.min(axis=1) // min_bins_per_rep).clip(lower=1, upper=K_max)
K_balanced = int(K_per_sample.min())
K = K_balanced
print(f"Balanced within-sample K = {K} (based on min bins across lesion/nonlesion per sample)")

# --- 3) build pseudo-replicate pseudo-bulks ---
X = adata_dge.X.tocsr() if sp.issparse(adata_dge.X) else np.asarray(adata_dge.X)
gene_names = adata_dge.var_names

rng = np.random.default_rng(0)

pb = []
meta_rows = []

for sample in adata_dge.obs[sample_key].astype(str).unique():
    for lesion in ["0", "1"]:
        mask = (adata_dge.obs[sample_key].astype(str) == str(sample)) & (adata_dge.obs[lesion_key] == lesion)
        idx = np.where(mask.values)[0]
        if len(idx) == 0:
            continue

        # For strict balance between lesion/nonlesion within sample, we may need to subsample
        # to exactly K * m bins. Choose m as floor(len(idx)/K)
        m = len(idx) // K
        if m == 0:
            raise ValueError(f"Not enough bins in sample={sample}, lesion={lesion} for K={K}")

        # Use exactly K*m bins (drops remainder) for equal-sized chunks
        idx = rng.permutation(idx)[: K * m]
        idx = idx.reshape(K, m)

        for r in range(K):
            s = X[idx[r]].sum(axis=0)
            s = np.asarray(s).ravel()
            pb_id = f"{sample}__{lesion}__rep{r+1:02d}"
            pb.append(pd.Series(s, name=pb_id, index=gene_names))
            meta_rows.append({ "pb_id": pb_id, sample_key: str(sample), lesion_key: lesion, "replicate": r+1 })

counts_df = pd.DataFrame(pb).astype(int)
meta = pd.DataFrame(meta_rows).set_index("pb_id")

print(counts_df.shape, meta.shape)
meta

dds = DeseqDataSet(
    counts=counts_df,
    metadata=meta,
    design_factors=[sample_key, lesion_key],
    refit_cooks=True,
)
dds.deseq2()

stat = DeseqStats(dds, contrast=[lesion_key, "1", "0"])
stat.summary()
res = stat.results_df

import gseapy as gp
#res_rank = res.dropna(subset=["stat"]).set_index("gene")
rnk = res["stat"].sort_values(ascending=False)  # + = up in lesion

gmt_dir = "/data/kanferg/Sptial_Omics/projects/NatalieLab/liver_cancer/spatialomicstoolkit/gmt"
gmt_files = [f for f in os.listdir(gmt_dir) if f.endswith('.gmt') and not f.startswith('._')]

all_results = {}
for gmt_file in gmt_files:
    gmt_path = os.path.join(gmt_dir, gmt_file)
    pre_res = gp.prerank(
        rnk=rnk,
        gene_sets=gmt_path,
        min_size=10,
        max_size=500,
        permutation_num=1000,
        seed=0,
        outdir=None,
        verbose=False
    )
    # Store the results dataframe
    all_results[gmt_file] = pre_res.res2d
    
import math
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# --- Configuration for Publication ---
FIG_WIDTH = 16          # Total figure width
ROW_HEIGHT = 6          # Height per row
FONT_SIZE_TITLE = 14
FONT_SIZE_LABEL = 12
FONT_SIZE_TICK = 10
DOT_SCALE_FACTOR = 15   

def clean_term_label(term):
    """Cleans pathway names for better readability."""
    term = str(term)
    prefixes_to_remove = ['GOBP_', 'GOCC_', 'GOMF_', 'HALLMARK_', 'HP_', 'MP_','GTRD_']
    for prefix in prefixes_to_remove:
        if term.startswith(prefix):
            term = term.replace(prefix, '')
            break
    return term.replace('_', ' ').capitalize()

def clean_plot_title(filename):
    """Maps GMT filenames to readable titles."""
    filename = filename.lower()
    if 'go.bp' in filename: return 'GO Biological Process'
    if 'go.cc' in filename: return 'GO Cellular Component'
    if 'go.mf' in filename: return 'GO Molecular Function'
    if 'mh.all' in filename: return 'Hallmark Pathways'
    if 'mpt' in filename: return 'Mammalian Phenotype'
    if 'gtrd' in filename: return 'TF Targets (GTRD)'
    return filename.replace('.gmt', '').replace('.', ' ').replace('_', ' ').title()

if all_results:
    n = len(all_results)
    cols = 2 
    # Add 1 to n to reserve a slot for the legend
    rows = math.ceil((n + 1) / cols)
    
    fig, axes = plt.subplots(rows, cols, figsize=(FIG_WIDTH, ROW_HEIGHT*rows), constrained_layout=True)
    axes = axes.flatten()
    
    for i, (gmt_name, df_orig) in enumerate(all_results.items()):
        ax = axes[i]
        
        # Filter: Top 10 by significance
        top_n = 10
        df = df_orig.sort_values("FDR q-val").head(top_n).copy()
        df = df.sort_values('NES', ascending=True)

        # Values
        cleaned_terms = [clean_term_label(t) for t in (df['Term'] if 'Term' in df.columns else df.index)]
        nes = df['NES']
        fdr = df['FDR q-val']
        
        # Calculate dynamic limits for this specific plot
        data_min = fdr.min()
        data_max = fdr.max()
        
        if data_max == data_min:
            c_min = 0.0
            c_max = max(data_max, 0.05) 
        else:
            c_min = 0.0 
            c_max = data_max 

        # Sizes
        if 'Gene %' in df.columns:
            sizes = df['Gene %'].astype(str).str.rstrip('%').astype(float)
        else:
            sizes = pd.Series([20.0]*len(df))
            
        # Scatter plot
        sc = ax.scatter(nes, cleaned_terms, c=fdr, s=sizes*DOT_SCALE_FACTOR, 
                        cmap='viridis_r', vmin=c_min, vmax=c_max,
                        edgecolor='black', linewidth=0.5, alpha=0.9, zorder=2) 
                        
        # Decorate
        readable_title = clean_plot_title(gmt_name)
        ax.set_title(readable_title, fontsize=FONT_SIZE_TITLE, fontweight='bold')
        ax.set_xlabel('Normalized Enrichment Score (NES)', fontsize=FONT_SIZE_LABEL)
        
        ax.tick_params(axis='y', labelsize=FONT_SIZE_TICK)
        ax.tick_params(axis='x', labelsize=FONT_SIZE_TICK)
        
        ax.grid(True, which='both', linestyle='--', linewidth=0.5, color='gray', alpha=0.5, zorder=1)
        ax.axvline(0, color='black', linestyle='-', linewidth=0.8, zorder=1)
        
        # Colorbar
        cbar = plt.colorbar(sc, ax=ax)
        cbar.set_label('FDR', fontsize=FONT_SIZE_LABEL)
        cbar.ax.tick_params(labelsize=FONT_SIZE_TICK)

    # --- Create Separate Legend Subplot ---
    # Use the next available slot for the legend
    legend_ax = axes[n]
    legend_ax.axis('off')

    # Create dummy handles for the legend
    legend_handles = []
    sizes_to_show = [10, 25, 50, 75] # Standard percentages to display
    for s in sizes_to_show:
        # Create empty scatter points with correct sizes for the legend
        h = legend_ax.scatter([], [], s=s*DOT_SCALE_FACTOR, 
                              c='gray', edgecolors='black', alpha=0.6, linewidth=0.5)
        legend_handles.append(h)

    legend_ax.legend(handles=legend_handles, 
                     labels=[f"{s}%" for s in sizes_to_show],
                     loc='center', 
                     title="Gene %", 
                     title_fontsize=FONT_SIZE_TITLE,
                     fontsize=FONT_SIZE_LABEL,
                     frameon=False, 
                     labelspacing=1.5,
                     borderpad=1)

    # Hide any remaining subplots
    for j in range(n + 1, len(axes)):
        axes[j].axis('off')
        
    plt.show()
else:
    print("No results found to plot.")
    
from gseapy import gseaplot

# 1. Search for "Lipid Droplet"
target_keyword = "LIPID_DROPLET" 
found_gmt = None
found_term = None

print(f"Searching for pathways containing '{target_keyword}'...")

for gmt_file, df in all_results.items():
    # Check 'Term' column or Index
    terms = df['Term'] if 'Term' in df.columns else df.index
    # Find match
    matches = [t for t in terms if target_keyword.upper() in str(t).upper()]
    if matches:
        found_gmt = gmt_file
        found_term = matches[0] # Take the first match found
        print(f"Found '{found_term}' in {found_gmt}")
        break

if found_gmt and found_term:
    # 2. Re-run GSEA for this specific GMT to get the object needed for plotting
    # We need the 'results' dictionary which contains hits and running ES
    print(f"Re-running GSEA for {found_gmt} to generate plot...")
    gmt_path = os.path.join(gmt_dir, found_gmt)
    
    pre_res = gp.prerank(
        rnk=rnk,
        gene_sets=gmt_path,
        min_size=10,
        max_size=500,
        permutation_num=100, # Faster, just for plotting
        seed=0,
        outdir=None,
        verbose=False
    )
    
    # 3. Plot
    if found_term in pre_res.results:
         print(f"Plotting: {found_term}")
         gseaplot(rank_metric=pre_res.ranking,
                  term=found_term,
                  **pre_res.results[found_term],
                  ofname=None # Display inline
                 )
         plt.show()
    else:
         print(f"Warning: '{found_term}' was found in initial scan but not in re-run. Check min/max_size parameters.")
else:
    print(f"No pathway matching '{target_keyword}' found in the analyzed results.")
    
def get_leading_edge_genes(gsea_res, term, ranking_metric, top_n=20):
    pathway_row = gsea_res.res2d.query(f'Term == "{term}"')
    lead_genes = pathway_row['Lead_genes'].iloc[0].split(';')
    top_genes_df = ranking_metric[ranking_metric.index.isin(lead_genes)].to_frame(name='Stat')
    return top_genes_df

top_genes = get_leading_edge_genes(pre_res, found_term, rnk)

sel_gens = np.intersect1d(top_genes.index, agg_adata_pairs.var_names).tolist()
adata_ld = agg_adata_pairs[:,sel_gens].copy()

import re

adata_lesion_only = adata_ld.copy()

print("Lesion bins per timepoint:")
print(adata_lesion_only.obs['sample'].value_counts().sort_index())

# 2. Generate Pseudo-bulk Replicates (Timepoints)
# We aggregate the lesion bins into balanced pseudo-replicates for e2, e4, e6

# Determine K (replicates) based on the smallest sample size
groups = adata_lesion_only.obs['sample'].astype(str)
group_counts = groups.value_counts()
min_group_n = int(group_counts.min())

min_bins_per_rep = 200  # Minimum bins to form a robust pseudo-bulk
K_max = 10               # Max replicates we want

K = max(1, min(K_max, min_group_n // min_bins_per_rep))
print(f"\nConfiguration: Smallest sample has {min_group_n} bins.")
print(f"Creating {K} pseudo-replicates per timepoint.")

# Matrix and Generator
X_lesion = adata_lesion_only.X.tocsr() if sp.issparse(adata_lesion_only.X) else np.asarray(adata_lesion_only.X)
gene_names = adata_lesion_only.var_names
rng = np.random.default_rng(42)

pb_list = []
meta_time_rows = []

for sample in ['exp_2', 'exp_4', 'exp_6']:
    # Get indices for this sample
    mask = (adata_lesion_only.obs['sample'] == sample)
    idx = np.where(mask.values)[0]
    
    # Calculate how many bins to use to get equal chunks
    m = len(idx) // K
    if m == 0: continue # Should be handled by K calc, but safe check
        
    # Shuffle and trim to exact multiple of K
    idx = rng.permutation(idx)[: K * m]
    idx = idx.reshape(K, m)
    
    for r in range(K):
        # Sum counts
        s = X_lesion[idx[r]].sum(axis=0)
        s = np.asarray(s).ravel()
        
        pb_id = f"{sample}_rep{r+1:02d}"
        pb_list.append(pd.Series(s, name=pb_id, index=gene_names))
        meta_time_rows.append({ "pb_id": pb_id, "sample": sample, "replicate": r+1 })

counts_time_df = pd.DataFrame(pb_list).astype(int)
meta_time = pd.DataFrame(meta_time_rows).set_index("pb_id")

print(f"Pseudo-bulk Matrix Shape: {counts_time_df.shape}")
meta_time.head()

# 3. Run DESeq2 (Design: ~sample)
dds_time = DeseqDataSet(
    counts=counts_time_df,
    metadata=meta_time,
    design_factors=['sample'], # Looking for differences between samples (timepoints)
    refit_cooks=True,
    n_cpus=1 # Changed from 4 to 1 to fix TerminatedWorkerError
)

print("Running DESeq2 (Single threaded)...")
dds_time.deseq2()
print("Done.")


# filepath: 
# ...existing code...

# 1. Extract Normalized Counts for Visualization
# pydeseq2 stores 'normed_counts' in layers after running deseq2()
if 'normed_counts' in dds_time.layers:
    norm_counts = dds_time.layers['normed_counts']
else:
    # Fallback: manually normalize using size factors if layer is missing
    norm_counts = dds_time.X / dds_time.obsm["size_factors"][:, None]

# Log-transform (log1p) to stabilize variance for plotting
log_counts = np.log1p(norm_counts)

# 2. Prepare DataFrame (Genes as rows, Samples as columns)
df_plot = pd.DataFrame(log_counts, index=dds_time.obs.index, columns=dds_time.var.index).T

# 3. Organize Columns by Timepoint (e2 -> e4 -> e6)
meta_sorted = meta_time.sort_values("sample")
df_plot = df_plot[meta_sorted.index]

# 4. Create Colors for Timepoints
# Map each sample group to a distinct color for the annotation bar
pal = sns.color_palette("Set2", n_colors=3)
sample_map = {'e2': pal[0], 'e4': pal[1], 'e6': pal[2]}
col_colors = meta_sorted['sample'].map(sample_map)

# 5. Generate Heatmap
# z_score=0: Computes z-score across rows (genes) directly within clustermap
g = sns.clustermap(
    df_plot,
    z_score=0,                 # Standardize rows (genes) to see relative trends
    cmap="vlag",               # Red-Blue diverging colormap (Blue=Low, Red=High)
    center=0,
    col_cluster=False,         # Keep samples ordered by time (e2->e4->e6)
    row_cluster=True,          # Cluster genes with similar expression profiles
    col_colors=col_colors,     # Add color bar for timepoints
    figsize=(10, 10),
    dendrogram_ratio=(0.2, 0.1),
    cbar_pos=(0.02, 0.8, 0.03, 0.15) # Position colorbar
)

# 6. Formatting
g.ax_heatmap.set_title(f"Expression Trend: {found_term}", y=1.2, fontsize=14, fontweight='bold')
g.ax_heatmap.set_xlabel("Replicates (Timepoint)", fontsize=12)
g.ax_heatmap.set_ylabel("Leading Edge Genes", fontsize=12)

# Create a custom legend for the timepoints
from matplotlib.patches import Patch
handles = [Patch(facecolor=sample_map[key], edgecolor='k', label=key) for key in ['e2', 'e4', 'e6']]
g.ax_col_dendrogram.legend(handles=handles, title="Timepoint", loc="center", bbox_to_anchor=(0.5, 1.5), ncol=3, frameon=False)

plt.show()

def run_gsea(adata_dge,exprement_samples):
    import numpy as np
    import pandas as pd
    import scipy.sparse as sp

    sample_key = "sample"
    lesion_key = "in_lesion"

    # 0/1 as strings for contrasts
    adata_dge.obs[lesion_key] = adata_dge.obs[lesion_key].astype(int).astype(str)

    # group label
    groups = adata_dge.obs[[sample_key, lesion_key]].astype(str).agg("__".join, axis=1)

    # --- 1) count bins per group ---
    group_counts = groups.value_counts().sort_index()
    print(group_counts)

    # --- 2) choose a balanced K based on smallest group ---
    min_bins_per_rep = 200   # adjust (100–500 depending on depth)
    K_max = 10

    min_group_n = int(group_counts.min())
    K = max(1, min(K_max, min_group_n // min_bins_per_rep))

    print(f"min group bins = {min_group_n}, using K = {K} pseudo-reps per (sample×lesion)")

    # If you want EXACT balance within each sample between lesion and non-lesion:
    # K_sample = min over samples of min(n_lesion, n_nonlesion) // min_bins_per_rep
    by_sample = (
        adata_dge.obs.assign(_group=groups.values)
        .groupby([sample_key, lesion_key])
        .size()
        .unstack(fill_value=0)
    )
    # compute per-sample feasible K (balanced between lesion/nonlesion)
    K_per_sample = (by_sample.min(axis=1) // min_bins_per_rep).clip(lower=1, upper=K_max)
    K_balanced = int(K_per_sample.min())
    K = K_balanced
    print(f"Balanced within-sample K = {K} (based on min bins across lesion/nonlesion per sample)")

    # --- 3) build pseudo-replicate pseudo-bulks ---
    X = adata_dge.X.tocsr() if sp.issparse(adata_dge.X) else np.asarray(adata_dge.X)
    gene_names = adata_dge.var_names

    rng = np.random.default_rng(0)

    pb = []
    meta_rows = []

    for sample in adata_dge.obs[sample_key].astype(str).unique():
        for lesion in ["0", "1"]:
            mask = (adata_dge.obs[sample_key].astype(str) == str(sample)) & (adata_dge.obs[lesion_key] == lesion)
            idx = np.where(mask.values)[0]
            if len(idx) == 0:
                continue

            # For strict balance between lesion/nonlesion within sample, we may need to subsample
            # to exactly K * m bins. Choose m as floor(len(idx)/K)
            m = len(idx) // K
            if m == 0:
                raise ValueError(f"Not enough bins in sample={sample}, lesion={lesion} for K={K}")

            # Use exactly K*m bins (drops remainder) for equal-sized chunks
            idx = rng.permutation(idx)[: K * m]
            idx = idx.reshape(K, m)

            for r in range(K):
                s = X[idx[r]].sum(axis=0)
                s = np.asarray(s).ravel()
                pb_id = f"{sample}__{lesion}__rep{r+1:02d}"
                pb.append(pd.Series(s, name=pb_id, index=gene_names))
                meta_rows.append({ "pb_id": pb_id, sample_key: str(sample), lesion_key: lesion, "replicate": r+1 })

    counts_df = pd.DataFrame(pb).astype(int)
    meta = pd.DataFrame(meta_rows).set_index("pb_id")

    dds = DeseqDataSet(
    counts=counts_df,
    metadata=meta,
    design_factors=[sample_key, lesion_key],
    refit_cooks=True,
    )
    dds.deseq2()

    stat = DeseqStats(dds, contrast=[lesion_key, "1", "0"])
    stat.summary()
    res = stat.results_df
    path_temp = '/data/kanferg/Sptial_Omics/projects/NatalieLab/liver_cancer/spatialomicstoolkit/temp/'
    res.to_csv(f"{path_temp}summary_DGE_Lesion_0vs1_version_3_{exprement_samples}.csv")
    import gseapy as gp
    rnk = res["stat"].sort_values(ascending=False)  # + = up in lesion
    gmt_dir = "/data/kanferg/Sptial_Omics/projects/NatalieLab/liver_cancer/spatialomicstoolkit/gmt"
    gmt_files = [f for f in os.listdir(gmt_dir) if f.endswith('.gmt') and not f.startswith('._')]

    all_results = {}
    for gmt_file in gmt_files:
        gmt_path = os.path.join(gmt_dir, gmt_file)
        pre_res = gp.prerank(
            rnk=rnk,
            gene_sets=gmt_path,
            min_size=10,
            max_size=500,
            permutation_num=1000,
            seed=0,
            outdir=None,
            verbose=False
        )
        # Store the results dataframe
        all_results[gmt_file] = pre_res.res2d
    import math
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors

    # --- Plot configuration tuned for notebook output ---
    PANEL_WIDTH = 8.5
    MIN_PANEL_HEIGHT = 4.8
    HEIGHT_PER_TERM = 0.42
    FONT_SIZE_TITLE = 14
    FONT_SIZE_LABEL = 12
    FONT_SIZE_TICK = 10
    DOT_SCALE_FACTOR = 15

    def clean_term_label(term):
        """Cleans pathway names for better readability."""
        term = str(term)
        prefixes_to_remove = ['GOBP_', 'GOCC_', 'GOMF_', 'HALLMARK_', 'HP_', 'MP_','GTRD_']
        for prefix in prefixes_to_remove:
            if term.startswith(prefix):
                term = term.replace(prefix, '')
                break
        term = term.replace('_', ' ').capitalize()
        if len(term) > 45:
            words = term.split()
            lines = []
            current = []
            current_len = 0
            for word in words:
                extra = len(word) + (1 if current else 0)
                if current_len + extra > 32:
                    lines.append(' '.join(current))
                    current = [word]
                    current_len = len(word)
                else:
                    current.append(word)
                    current_len += extra
            if current:
                lines.append(' '.join(current))
            term = '\n'.join(lines)
        return term

    def clean_plot_title(filename):
        """Maps GMT filenames to readable titles."""
        filename = filename.lower()
        if 'go.bp' in filename: return 'GO Biological Process'
        if 'go.cc' in filename: return 'GO Cellular Component'
        if 'go.mf' in filename: return 'GO Molecular Function'
        if 'mh.all' in filename: return 'Hallmark Pathways'
        if 'mpt' in filename: return 'Mammalian Phenotype'
        if 'gtrd' in filename: return 'TF Targets (GTRD)'
        return filename.replace('.gmt', '').replace('.', ' ').replace('_', ' ').title()

    if all_results:
        n = len(all_results)
        cols = min(3, n)
        rows = math.ceil(n / cols)
        top_n = 10
        panel_height = max(MIN_PANEL_HEIGHT, top_n * HEIGHT_PER_TERM + 1.8)
        fig_width = cols * PANEL_WIDTH + 2.0
        fig_height = rows * panel_height + 1.0

        fig, axes = plt.subplots(
            rows,
            cols,
            figsize=(fig_width, fig_height),
            constrained_layout=True,
        )
        axes = np.atleast_1d(axes).flatten()
        
        for i, (gmt_name, df_orig) in enumerate(all_results.items()):
            ax = axes[i]
            
            # Filter: Top 10 by significance
            df = df_orig.sort_values("FDR q-val").head(top_n).copy()
            df = df.sort_values('NES', ascending=True)

            # Values
            cleaned_terms = [clean_term_label(t) for t in (df['Term'] if 'Term' in df.columns else df.index)]
            nes = df['NES']
            fdr = df['FDR q-val']
            
            # Calculate dynamic limits for this specific plot
            data_min = fdr.min()
            data_max = fdr.max()
            
            if data_max == data_min:
                c_min = 0.0
                c_max = max(data_max, 0.05) 
            else:
                c_min = 0.0 
                c_max = data_max 

            # Sizes
            if 'Gene %' in df.columns:
                sizes = df['Gene %'].astype(str).str.rstrip('%').astype(float)
            else:
                sizes = pd.Series([20.0]*len(df))
                
            # Scatter plot
            sc = ax.scatter(nes, cleaned_terms, c=fdr, s=sizes*DOT_SCALE_FACTOR, 
                            cmap='viridis_r', vmin=c_min, vmax=c_max,
                            edgecolor='black', linewidth=0.5, alpha=0.9, zorder=2) 
                            
            # Decorate
            readable_title = clean_plot_title(gmt_name)
            ax.set_title(readable_title, fontsize=FONT_SIZE_TITLE, fontweight='bold')
            ax.set_xlabel('Normalized Enrichment Score (NES)', fontsize=FONT_SIZE_LABEL)
            
            ax.tick_params(axis='y', labelsize=FONT_SIZE_TICK)
            ax.tick_params(axis='x', labelsize=FONT_SIZE_TICK)
            
            ax.grid(True, which='both', linestyle='--', linewidth=0.5, color='gray', alpha=0.5, zorder=1)
            ax.axvline(0, color='black', linestyle='-', linewidth=0.8, zorder=1)
            
            # Colorbar
            cbar = plt.colorbar(sc, ax=ax, fraction=0.055, pad=0.03)
            cbar.set_label('FDR', fontsize=FONT_SIZE_LABEL)
            cbar.ax.tick_params(labelsize=FONT_SIZE_TICK)
            ax.margins(y=0.15)
            ax.set_facecolor('#fbfbfb')

        # --- Create Figure-Level Size Legend ---
        legend_handles = []
        sizes_to_show = [10, 25, 50, 75] # Standard percentages to display
        for s in sizes_to_show:
            h = plt.scatter([], [], s=s*DOT_SCALE_FACTOR, 
                                c='gray', edgecolors='black', alpha=0.6, linewidth=0.5)
            legend_handles.append(h)

        fig.legend(handles=legend_handles, 
                   labels=[f"{s}%" for s in sizes_to_show],
                   loc='lower center', 
                   bbox_to_anchor=(0.5, -0.02),
                   ncol=len(sizes_to_show),
                   title="Gene %", 
                   title_fontsize=FONT_SIZE_TITLE,
                   fontsize=FONT_SIZE_LABEL,
                   frameon=False, 
                   handletextpad=1.0,
                   columnspacing=1.8)

        # Hide any remaining subplots
        for j in range(n, len(axes)):
            axes[j].axis('off')
            
        plt.show()
    else:
        print("No results found to plot.")
        
exprement_samples = ['exp_2']
adata_dge = agg_adata_pairs[agg_adata_pairs.obs['sample'].isin(exprement_samples)].copy()
adata_dge.obs.rename(columns={'in_lesions':'in_lesion'}, inplace=True)

run_gsea(adata_dge,exprement_samples = exprement_samples[0])

# pathway_campute_aucell
'''
path = "/data/kanferg/Sptial_Omics/projects/NatalieLab/liver_cancer/spatialomicstoolkit/data_out/EXPORT_ENRICHMENTMAP_COMBINED/"
cytoscape_df = pd.read_csv(os.path.join(path, "table_key_man_cluster.csv"))


gmt_dir = "/data/kanferg/Sptial_Omics/projects/NatalieLab/liver_cancer/spatialomicstoolkit/gmt"
gmt_files = [f for f in os.listdir(gmt_dir) if f.endswith(".gmt")]
'''
def gmt_to_decoupler(file) -> pd.DataFrame:
    """Parse a gmt file to a decoupler pathway dataframe."""
    from itertools import chain, repeat

    pathways = {}

    with open(file,"r") as f:
        for line in f:
            name, _, *genes = line.strip().split("\t")
            pathways[name] = genes

    return pd.DataFrame.from_records(
        chain.from_iterable(zip(repeat(k), v) for k, v in pathways.items()),
        columns=["geneset", "genesymbol"],
    )
gmt_dir = "/data/kanferg/Sptial_Omics/projects/NatalieLab/liver_cancer/spatialomicstoolkit/gmt"   
HK = gmt_to_decoupler(file = os.path.join(gmt_dir, 'mh.all.v2026.1.Mm.symbols.gmt'))
CP = gmt_to_decoupler(file = os.path.join(gmt_dir, 'm5.go.cc.v2026.1.Mm.symbols.gmt'))  
GTRD = gmt_to_decoupler(file = os.path.join(gmt_dir, 'm3.gtrd.v2026.1.Mm.symbols.gmt')) 

selected_pathways = {
    "HK":['HALLMARK_FATTY_ACID_METABOLISM','HALLMARK_E2F_TARGETS','HALLMARK_G2M_CHECKPOINT','HALLMARK_MITOTIC_SPINDLE','HALLMARK_MTORC1_SIGNALING'],
    "CP":['GOCC_LIPID_DROPLET','GOCC_SPINDLE','GOCC_SPINDLE_MICROTUBULE','GOCC_PROTON_TRANSPORTING_TWO_SECTOR_ATPASE_COMPLEX_PROTON_TRANSPORTING_DOMAIN','GOCC_PROTON_TRANSPORTING_V_TYPE_ATPASE_COMPLEX','GOCC_PROTON_TRANSPORTING_V_TYPE_ATPASE_V0_DOMAIN'],
    "GTRD":['ING1_TARGET_GENES','MSX1_TARGET_GENES','ZFP991_TARGET_GENES','NFE2_TARGET_GENES']
}

def removepathways(df, pathways):
    return df[df['geneset'].isin(pathways)]
HK = removepathways(HK, selected_pathways['HK'])
CP = removepathways(CP, selected_pathways['CP'])    
GTRD = removepathways(GTRD, selected_pathways['GTRD'])

def campute_pathway_score(andata,dataset ,oncogenic_pathways):
    decoupler.run_aucell(
        andata,
        dataset,
        source="geneset",
        target="genesymbol",
        use_raw=False,
        verbose=True,
    )
    andata.obs[oncogenic_pathways] = andata.obsm["aucell_estimate"][oncogenic_pathways]
    
    # upload cdata
andata_e6 = sc.read_h5ad('/data/kanferg/Sptial_Omics/projects/NatalieLab/liver_cancer/spatialomicstoolkit/out_1/bin2cell/for_spatialdata/SCAF4332_24008775_D1_VHD_cdata.h5ad')

campute_pathway_score(andata_e6,HK ,selected_pathways['HK'])
campute_pathway_score(andata_e6,GTRD ,selected_pathways['GTRD'])
campute_pathway_score(andata_e6,CP ,selected_pathways['CP'])

pathway_cols = ['HALLMARK_FATTY_ACID_METABOLISM',
        'HALLMARK_E2F_TARGETS', 'HALLMARK_G2M_CHECKPOINT',
        'HALLMARK_MITOTIC_SPINDLE', 'HALLMARK_MTORC1_SIGNALING',
        'GOCC_LIPID_DROPLET', 'GOCC_SPINDLE', 'GOCC_SPINDLE_MICROTUBULE',
        'GOCC_PROTON_TRANSPORTING_TWO_SECTOR_ATPASE_COMPLEX_PROTON_TRANSPORTING_DOMAIN',
        'GOCC_PROTON_TRANSPORTING_V_TYPE_ATPASE_COMPLEX',
        'GOCC_PROTON_TRANSPORTING_V_TYPE_ATPASE_V0_DOMAIN', 'ING1_TARGET_GENES',
        'MSX1_TARGET_GENES', 'ZFP991_TARGET_GENES', 'NFE2_TARGET_GENES']

df_aucell = pd.DataFrame({'object_id': andata_e6.obs['object_id'].astype(float)})
for col in pathway_cols:
    df_aucell[col] = andata_e6.obs[col].values

# composition analysis

In [ ]:
# xenium melanoma

def ct_table(adata_concat, ct_key='cell_type'):
    '''
    Generates a dataframe for cell type abundance analysis across conditions and batches.

    This function filters the input AnnData object for specific batches and conditions,
    then calculates the abundance of a specified cell type relative to the total number
    of cells per treatment-batch group. Pseudo-counts are added to avoid zero values.

    Parameters
    ----------
    adata_concat : AnnData
        The input AnnData object containing single-cell data.
    ct_key : str, default 'cell_type'
        The specific cell type name (found in adata.obs['cell_type']) to analyze.
        
    Returns
    -------
    pd.DataFrame
        A DataFrame with columns:
        - 'treatment_batch': Combined condition, day, and batch.
        - 'n_cells': Total cells in the group.
        - 'treatment': Combined condition and day.
        - 'n_cells_ct': Count of the specific cell type (with pseudo-count +1).
        - 'batch': Batch identifier.
        - 'celltype': The name of the cell type analyzed.
    '''
    adata_concat.obs['Condition'] = adata_concat.obs['Condition'].astype(str)
    batch_l = ['58','33','32','9']
    adata_concat = adata_concat[~adata_concat.obs['batch'].isin(batch_l)]
    adata_concat_rest = adata_concat[adata_concat.obs['Condition'].isin(['Pmel1','Pmel1.Il21', 'Pmel1.Il15', 'Pmel1.Il21.Il15', 'Mock'])].copy() 
    df = adata_concat_rest.obs.copy()
    def catTostring(colList):
        for col in colList:
            df[col] = df[col].astype(str)
    colList = ['cell_type','batch','Harvest_Day', 'Condition']
    catTostring(colList)
    df['treatment'] = df['Condition'] + '_' + df['Harvest_Day']
    df.loc[df['cell_type']==ct_key].groupby('treatment').size()
    df['treatment_batch'] = df['Condition'] + '_' + df['Harvest_Day'] + '_' + df['batch']
    df_plt = pd.DataFrame({'treatment_batch':df.groupby('treatment_batch').size().index.values, 'n_cells':df.groupby('treatment_batch').size(),"treatment":df.groupby(['treatment_batch','treatment'], as_index=False)['treatment'].size()['treatment'].values}).reset_index(drop=True)
    df_plt['n_cells_ct'] = df.loc[df['cell_type']==ct_key].groupby('treatment_batch').size().reset_index(drop=True)
    cond_order = ['Mock','Pmel1', 'Pmel1.Il21', 'Pmel1.Il15', 'Pmel1.Il21.Il15']
    day_order =['4','8']
    treatment_order = [f"{c}_{d}" for c in cond_order for d in day_order]
    df_plt["treatment"] = pd.Categorical(
        df_plt["treatment"],
        categories=treatment_order,
        ordered=True,
    )
    df_plt['batch'] = [re.sub(r'.*_', '', b) for b in df_plt['treatment_batch']]
    # add psudo counts to avoid zero counts
    df_plt['n_cells_ct'] = df_plt['n_cells_ct'] + 1
    df_plt['celltype'] = ct_key
    return df_plt

def _pick_col(adata, candidates):
    for c in candidates:
        if c in adata.obs.columns:
            return c
    raise KeyError(f"None of these columns were found in adata.obs: {candidates}")

def _get_ct_color_map(adata, ct_key="cell_type", colors_uns_key="cell_type_colors"):
    """
    Returns dict: {cell_type: color} using adata.uns[colors_uns_key] aligned to
    adata.obs[ct_key].cat.categories (Scanpy convention).
    """
    if colors_uns_key not in adata.uns:
        raise KeyError(f"'{colors_uns_key}' not found in adata.uns. Available keys: {list(adata.uns.keys())[:20]} ...")

    ct = adata.obs[ct_key]
    if not pd.api.types.is_categorical_dtype(ct):
        ct = ct.astype("category")

    cats = list(ct.cat.categories)
    cols = list(adata.uns[colors_uns_key])

    if len(cols) < len(cats):
        raise ValueError(
            f"adata.uns['{colors_uns_key}'] has {len(cols)} colors but {len(cats)} categories in obs['{ct_key}']."
        )

    return {k: cols[i] for i, k in enumerate(cats)}

def plot_celltype_stacked_100(
    adata,
    legend_order,
    ct_key="cell_type",
    condition_key="Condition",
    day_key_candidates=("harvest_day", "Harvest_Day"),
    batch_key_candidates=("batch", "tech_rep"),
    colors_uns_key="cell_type_colors",
    conditions_keep=("Pmel1", "Pmel1.Il21", "Pmel1.Il15", "Pmel1.Il21.Il15", "Mock"),
    exclude_batches=("58","33","32","9"),
    make_separate_figs=True,
):
    """
    Stacked bars normalized to 100% per Condition, split by day (two panels: day 4 and day 8).
    Uses colors from adata.uns[colors_uns_key] (Scanpy convention).
    """
    day_key = _pick_col(adata, list(day_key_candidates))
    # batch key is optional; only used for excluding batches if present
    batch_key = None
    for c in batch_key_candidates:
        if c in adata.obs.columns:
            batch_key = c
            break

    # copy obs only
    df = adata.obs[[ct_key, condition_key, day_key] + ([batch_key] if batch_key else [])].copy()
    df[condition_key] = df[condition_key].astype(str)
    df[day_key] = df[day_key].astype(str)
    df[ct_key] = df[ct_key].astype(str)

    # filter conditions
    if conditions_keep is not None:
        df = df[df[condition_key].isin(list(conditions_keep))].copy()

    # exclude specific batches if possible
    if batch_key is not None and exclude_batches:
        df = df[~df[batch_key].astype(str).isin(list(exclude_batches))].copy()

    # enforce x-order (Condition)
    cond_order = list(conditions_keep) if conditions_keep is not None else sorted(df[condition_key].unique())
    df[condition_key] = pd.Categorical(df[condition_key], categories=cond_order, ordered=True)

    # color map from uns
    ct_color_map = _get_ct_color_map(adata, ct_key=ct_key, colors_uns_key=colors_uns_key)

    # final legend order (only keep those that exist)
    present_cts = set(df[ct_key].unique())
    legend_order_present = [ct for ct in legend_order if ct in present_cts]

    # days to plot (prefer 4 then 8 if present)
    day_vals = list(pd.unique(df[day_key]))
    preferred_days = [d for d in ["4", "8"] if d in day_vals]
    other_days = [d for d in day_vals if d not in preferred_days]
    days_to_plot = preferred_days + sorted(other_days)

    # publication-ish styling
    plt.rcParams.update({
        "font.size": 7,
        "axes.titlesize": 8,
        "axes.labelsize": 7,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    def _make_panel(ax, d):
        ddf = df[df[day_key] == d].copy()

        # counts per (Condition, cell_type)
        tab = pd.crosstab(ddf[condition_key], ddf[ct_key])  # rows=Condition, cols=cell_type
        tab = tab.reindex(cond_order)  # enforce x order

        # keep only legend-ordered columns (and only those present)
        tab = tab.reindex(columns=legend_order_present, fill_value=0)

        # normalize to 100%
        denom = tab.sum(axis=1).replace(0, np.nan)
        frac = (tab.div(denom, axis=0) * 100).fillna(0)

        x = np.arange(frac.shape[0])
        bottom = np.zeros(frac.shape[0], dtype=float)

        for ct in frac.columns:
            ax.bar(
                x,
                frac[ct].values,
                bottom=bottom,
                width=0.85,
                color=ct_color_map.get(ct, "#999999"),
                edgecolor="white",
                linewidth=0.3,
                label=ct,
            )
            bottom += frac[ct].values

        ax.set_ylim(0, 100)
        ax.set_ylabel("Cell type (%)")
        ax.set_title(f"Day {d}")
        ax.set_xticks(x)
        ax.set_xticklabels([str(c) for c in frac.index], rotation=35, ha="right")
        ax.grid(axis="y", linewidth=0.3, alpha=0.3)
        ax.set_axisbelow(True)
        return ax

    if make_separate_figs:
        figs = []
        for d in days_to_plot[:2]:  # requested day 4 and day 8; will plot up to first two days found
            fig, ax = plt.subplots(figsize=(3.35, 2.6))  # single-column friendly
            _make_panel(ax, d)

            # legend outside
            handles, labels = ax.get_legend_handles_labels()
            ax.legend(
                handles, labels,
                title=None,
                loc="center left",
                bbox_to_anchor=(1.02, 0.5),
                frameon=False,
            )

            fig.tight_layout()
            figs.append(fig)
        return figs

    else:
        # two panels in one figure (day 4 + day 8)
        n = min(2, len(days_to_plot))
        fig, axes = plt.subplots(1, n, figsize=(6.8, 2.6), sharey=True)  # double-column width-ish
        if n == 1:
            axes = [axes]
        for ax, d in zip(axes, days_to_plot[:n]):
            _make_panel(ax, d)

        handles, labels = axes[-1].get_legend_handles_labels()
        fig.legend(
            handles, labels,
            loc="center left",
            bbox_to_anchor=(1.01, 0.5),
            frameon=False,
        )
        fig.tight_layout()
        return fig
    
    legend_order = [
    "Cytotoxic T-cells",
    "Proliferating T-cells",
    "Exhausting T-cells",
    "M1 TAM",
    "M2 TAM",
    "Macrophages",
    "MDSCs",
    "DC",
    "Endothelial cells",
    "ECM",
    "CAFs",
    "Tumor",
]

# make two separate figures: day 4 and day 8 (if present)
figs = plot_celltype_stacked_100(
    adata,
    legend_order=legend_order,
    ct_key="cell_type",
    condition_key="Condition",
    colors_uns_key="cell_type_colors",   # uses adata.uns['cell_type_colors']
    make_separate_figs=True,
)

# Save (optional)
figs[0].savefig("stacked_celltype_day4.pdf", bbox_inches="tight")
figs[1].savefig("stacked_celltype_day8.pdf", bbox_inches="tight")
plt.show()

#sccoda
'''
path_tables = "path_to_your_tables_directory"
path_tables = "/data/kanferg/Sptial_Omics/projects/NguyenLab/spatialomicstoolkit/article_notebooks"
df_in = pd.read_csv(os.path.join(path_tables, "sccoda_tumor_regions_input.csv"))
'''
class scooda_region_composition:
    ROI_BASE  = "core"        # compare border vs core (core is baseline)
    ROI_OTHER = "border"
    # --- choose reference cell type for scCODA log-ratio (must exist in cluster) ---
    REF_CELLTYPE = "Tumor"
    # --- HDI probability ---
    HDI_PROB = 0.94
    # --- HMC settings ---
    hmc_draws = 2000
    hmc_tune = 1000
    chains = 2
    target_accept = 0.9
    
    def __init__(self, df_input,BASE_COND = "IL-15#4"):
        self.df_input = df_input
        self.BASE_COND =BASE_COND    # compare everything relative to this
        self.ROI_BASE  = "core"        # compare border vs core (core is baseline)
        self.ROI_OTHER = "border"
        # --- choose reference cell type for scCODA log-ratio (must exist in cluster) ---
        self.REF_CELLTYPE = "Tumor"
        # --- HDI probability ---
        self.HDI_PROB = 0.94
        # --- HMC settings ---
        self.hmc_draws = 2000
        self.hmc_tune = 1000
        self.chains = 2
        self.target_accept = 0.9
        self.data,self.res = self.run_sccooda()
    
    @staticmethod
    def area_logOddsRatio(df_arr):
        df_border = df_arr.query('roi=="border"')
        df_core = df_arr.query('roi=="core"')
        frac =  df_border['roi_area'].iloc[0]/df_core['roi_area'].iloc[0]
        return np.log(frac) 

    def logit(frac):
        return np.log(frac/(1-frac))
    
    @staticmethod
    def compute_param(df_curr,df_immun,ct):
        def logit(frac):
            return np.log(frac/(1-frac))
        if df_curr.empty or (len(df_curr.roi.unique())!=2):
            cell = ct
            df_ct = df_immun.query(f'cluster==@cell')
            total = int(np.mean(df_ct.groupby(['batch']).size().values))
            border = int(np.mean(df_ct.query('roi=="border"').groupby(['batch']).size().values))
            core = int(np.mean(df_ct.query('roi=="core"').groupby(['batch']).size().values))
        else:
            total = len(df_curr)
            border = len(df_curr.query('roi == "border"'))
            core = len(df_curr.query('roi == "core"'))
        frac_border = border / total
        if  frac_border==0.0:
            frac_border = 1e-5
        logit_border = logit(frac_border)
        frac_core = core / total
        if  frac_core==0.0:
            frac_core = 1e-5
        logit_core = logit(frac_core)
        loddr_roi = logit_border - logit_core
        return total, border, core, loddr_roi
    
    
    
    @staticmethod
    def add_fdr_flag(df_effects, res_obj, fdr=0.05):
        """
        Merge scCODA effect_df (after set_fdr) into df_effects using
        (celltype, covariate). Returns df with final_parameter + credible flag.
        """
        res_obj.set_fdr(est_fdr=fdr)

        eff = res_obj.effect_df.copy().reset_index()

        # Normalize expected column names across versions
        rename_map = {}
        for col in eff.columns:
            if col.lower() == "covariate":
                rename_map[col] = "covariate"
            if col.lower() in ["cell type", "cell_type", "celltype"]:
                rename_map[col] = "celltype"
            if col.lower() in ["final parameter", "final_parameter"]:
                rename_map[col] = "final_parameter"

        eff = eff.rename(columns=rename_map)

        if not set(["covariate", "celltype", "final_parameter"]).issubset(set(eff.columns)):
            raise ValueError(
                "Could not find required columns in res.effect_df.\n"
                f"Columns present: {list(eff.columns)}"
            )

        out = df_effects.merge(
            eff[["covariate", "celltype", "final_parameter"]],
            on=["covariate", "celltype"],
            how="left"
        )

        out[f"credible_FDR{int(fdr*100):02d}"] = out["final_parameter"].fillna(0) != 0
        return out
    
    def run_sccooda(self):
        df = self.df_input.copy()

        # Normalize ROI labels to exactly "core" / "border"
        df["roi"] = (
            df["roi"]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        # (optional) map common variants to core/border if you have them
        roi_map = {
            "Core": "core",
            "Border": "border",
            "CORE": "core",
            "BORDER": "border",
            "core ": "core",
            "border ": "border",
        }
        df["roi"] = df["roi"].replace(roi_map)

        # Make sample ID
        df["sample"] = df["batch"].astype(str) + "_" + df["roi"].astype(str)

        # Covariates (1 row per sample)
        cov = (
            df[["sample", "Condition_day", "roi"]]
            .drop_duplicates(subset=["sample"])
            .set_index("sample")
        )

        # Count table: sample x cluster
        Y = pd.crosstab(df["sample"], df["cluster"].astype(str))

        # Merge covariates + counts
        df_model = cov.join(Y, how="left").fillna(0)

        count_cols = Y.columns.tolist()
        df_model[count_cols] = df_model[count_cols].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)

        # Drop empty samples (safety)
        df_model = df_model.loc[df_model[count_cols].sum(axis=1) > 0].copy()


        # ------------------------------------------------------------
        # 2) Build scCODA data object
        # ------------------------------------------------------------
        data = ccd.from_pandas(df_model, covariate_columns=["Condition_day", "roi"])

        # ------------------------------------------------------------
        # 3) Fit scCODA model:
        # Condition_day * roi
        # with baseline = BASE_COND and roi baseline = ROI_BASE
        # ------------------------------------------------------------
        formula = (
            f"C(Condition_day, Treatment('{self.BASE_COND}')) * "
            f"C(roi, Treatment('{self.ROI_BASE}'))"
        )

        model = ca.CompositionalAnalysis(
            data,
            formula=formula,
            reference_cell_type='Tumor'
        )

        # --- HMC settings ---
        hmc_draws = 2000
        hmc_tune = 1000
        chains = 2
        target_accept = 0.9

        # Handle scCODA version differences in sample_hmc signature
        sig = inspect.signature(model.sample_hmc)
        kwargs = {}

        if "num_results" in sig.parameters:
            kwargs["num_results"] = self.hmc_draws
        if "num_burnin" in sig.parameters:
            kwargs["num_burnin"] = self.hmc_tune
        if "num_warmup" in sig.parameters:
            kwargs["num_warmup"] = self.hmc_tune
        if "target_accept_prob" in sig.parameters:
            kwargs["target_accept_prob"] = self.target_accept
        if "target_accept" in sig.parameters:
            kwargs["target_accept"] = self.target_accept
        if "chains" in sig.parameters:
            kwargs["chains"] = self.chains
        elif "n_chains" in sig.parameters:
            kwargs["n_chains"] = self.chains

        res = model.sample_hmc(**kwargs)
        return data,res
    
    def table_generate(self):
        res = self.res
        data = self.data
        beta = res.posterior["beta"]
        cov_names = [str(c) for c in beta.coords["covariate"].values]
        cell_types = list(beta.coords["cell_type"].values)

        def interaction_cov_name(level, base=self.BASE_COND, roi_ref=self.ROI_BASE, roi_other=self.ROI_OTHER):
            return (
                f"C(Condition_day, Treatment('{base}'))[T.{level}]:"
                f"C(roi, Treatment('{roi_ref}'))[T.{roi_other}]"
            )

        # pick which condition levels to report (exclude the baseline itself)
        all_levels = sorted(data.obs["Condition_day"].unique())
        levels = [x for x in all_levels if x != self.BASE_COND]

        rows = []
        for ct in cell_types:
            for level in levels:
                term = interaction_cov_name(level)

                # some terms might be missing if data is sparse; skip safely
                if term not in cov_names:
                    continue

                draws = beta.sel(cell_type=ct, covariate=term).values.ravel()
                hdi = az.hdi(draws, hdi_prob=self.HDI_PROB)

                rows.append({
                    "celltype": str(ct),
                    "level": str(level),
                    "contrast": f"{level} minus {self.BASE_COND}",
                    "covariate": term,
                    "delta_mean": float(draws.mean()),
                    "delta_median": float(np.median(draws)),
                    "hdi_lo": float(hdi[0]),
                    "hdi_hi": float(hdi[1]),
                    "P_gt0": float((draws > 0).mean()),
                    "P_lt0": float((draws < 0).mean()),
                })

        df_interaction = pd.DataFrame(rows).sort_values(["celltype", "level"]).reset_index(drop=True)
        df_final = self.add_fdr_flag(df_interaction, res, fdr=0.05)
        eff = res.effect_df.copy().reset_index()
        
        rename_map = {}

        for col in eff.columns:
            cl = col.lower()
            if cl == "covariate":
                rename_map[col] = "covariate"
            elif cl in ["cell type", "cell_type", "celltype"]:
                rename_map[col] = "celltype"
            elif ("inclusion" in cl and "prob" in cl) or ("inclusion probability" in cl):
                rename_map[col] = "inclusion_prob"
            elif "posterior" in cl and "inclusion" in cl:
                rename_map[col] = "inclusion_prob"

        eff = eff.rename(columns=rename_map)

        if "inclusion_prob" not in eff.columns:
            raise ValueError(
                "Could not find inclusion probability in res.effect_df.\n"
                f"Columns present: {list(eff.columns)}\n"
                "Try printing res.effect_df.head() and we will map the exact column name."
            )

        # Merge inclusion probability into df_final
        df_final = df_final.merge(
            eff[["covariate", "celltype", "inclusion_prob"]],
            on=["covariate", "celltype"],
            how="left"
        )

        # local FDR proxy (per-effect): 1 - inclusion_prob
        df_final["local_fdr"] = 1.0 - df_final["inclusion_prob"]
        # Bayesian q-value style number:
        # sort by strongest evidence first (largest inclusion_prob)
        tmp = df_final.sort_values("inclusion_prob", ascending=False).copy()
        tmp["fdr_at_threshold"] = tmp["local_fdr"].expanding().mean()
        tmp["q_value"] = tmp["fdr_at_threshold"][::-1].cummin()[::-1]

        # merge back
        df_final = df_final.merge(
            tmp[["covariate", "celltype", "q_value", "fdr_at_threshold"]],
            on=["covariate", "celltype"],
            how="left"
        )
        df_final = df_final[['celltype', 'level', 'delta_mean', 'hdi_lo', 'hdi_hi', 'local_fdr', 'q_value']]
        return df_final
    
    def raw_table(self):
        df_immun = self.df_input.copy()
        table_key = df_immun[['Condition','batch','Harvest_Day']]
        table_key.drop_duplicates(['batch'],inplace=True)
        b2cond = {b:cond for b,cond in zip(table_key['batch'],table_key['Condition'])}
        b2day = {b:day for b,day in zip(table_key['batch'],table_key['Harvest_Day'])} 
        clusters = df_immun['cluster'].unique()
        container = []  
        for batch in df_immun.batch.astype(str).unique() :
            for cells in clusters:
                df_curr = df_immun.query('batch==@batch and cluster==@cells')
                total, border, core, loddr_roi = self.compute_param(df_curr,df_immun,cells)
                container.append(pd.DataFrame({'cell':[cells],
                                        'batch':[batch],
                                        'total':[total],
                                        'border':[border],
                                        'core':[core],
                                        # 'logodds_area':[logOddsArea],
                                        'loddr_roi':[loddr_roi],}))
        df_values = pd.concat(container).reset_index()
        df_values['harvest_day'] = df_values['batch'].map(b2day)
        df_values['condition'] = df_values['batch'].map(b2cond)
        return df_values
    
scooda_4_obj = scooda_region_composition(df_input = df_in,BASE_COND = "IL-15#4")
df_sum_4 = scooda_4_obj.table_generate()
df_raw_4 = scooda_4_obj.raw_table()
scooda_8_obj = scooda_region_composition(df_input = df_in,BASE_COND = "IL-15#8")
df_sum_8 = scooda_8_obj .table_generate()
df_raw_8 = scooda_8_obj.raw_table()

df_sum_4['baseline'] = 'IL-15#4'
df_sum_8['baseline'] = 'IL-15#8'
df_summary = pd.concat([df_sum_4, df_sum_8], axis=0)    

#- Dotplot of top enriched pathways
portal_central_markers = {
            'portal': ["Alb", "Aldob", "Arg1", "Asl", "Cyp2f2", "Fbp1", "Hal", "Hpx", 
                       "Hsd17b13", "Mup20", "Pck1", "Slc25a47", "Trf", "Ass1"],
            'central': ["Glul","Akr1c6", "Aldh1a1", "Car3", "Ces1c", "Cyp1a2", "Cyp2c37", "Cyp2c50", 
                        "Cyp2e1", "Gsta3", "Mgst1",  "Mup17", "Oat", 
                        "Pon1", "Rgn"]}
sc.pl.dotplot(adata_dot[adata_dot.obs['sample']=='exp_6'], portal_central_markers, groupby="cell_type", standard_scale='var',log = True,title='Week 6')



# Spatial distances / neighbors

In [ ]:
# moran I
'''
agg_adata_pairs = sc.read_h5ad('/data/HiTIF/data/spatialomics/liver_cancer/data/models/comp_analysis/cell_assign_pair_sample_from_2umbins_zone_report_21.h5ad')
agg_adata_pairs.obs.rename(columns={'in_lesions':'in_lesion'}, inplace=True)
'''
andata_e6 = agg_adata_pairs[agg_adata_pairs.obs['sample'] == 'exp_6'].copy()
andata_e6.layers['log_voyager'] = andata_e6.X.copy()
sc.pp.scale(andata_e6, max_value=10)
sc.pp.pca(andata_e6, n_comps=20, random_state=1337)
adata = andata_e6.copy()
adata.X = adata.layers['log_voyager'].copy()

sc.pp.neighbors(
    adata,
    n_neighbors=60,
    n_pcs=20,
    use_rep='X_pca',
    knn=True,
    random_state=1948,
    key_added='knn'
)
dist = adata.obsp['knn_distances'].copy()
dist.data = 1 / dist.data

# row normalize the matrix, this makes the matrix dense.
dist /= dist.sum(axis=1)

# convert dist back to sparse matrix
from scipy.sparse import csr_matrix
adata.obsp["knn_weights"] = csr_matrix(dist)

del dist

knn_graph = "knn_weights"

# adata.obsp["knn_connectivities"] represent the edges, while adata.opsp["knn_weights"] represent the weights
adata.obsp["knn_connectivities"] = (adata.obsp[knn_graph] > 0).astype(int)
vp.spatial.set_default_graph(adata, knn_graph)
vp.spatial.to_spatial_weights(adata, graph_name=knn_graph)

pathways = ["GTRD::NFE2_TARGET_GENES","Hk::HALLMARK_FATTY_ACID_METABOLISM","Hk::HALLMARK_PEROXISOME","BP::GOBP_LIPID_DROPLET_ORGANIZATION","Hk::HALLMARK_MTORC1_SIGNALING"]
#pathways = ["Hk::HALLMARK_FATTY_ACID_METABOLISM","Hk::HALLMARK_PEROXISOME","BP::GOBP_LIPID_DROPLET_ORGANIZATION","Hk::HALLMARK_MTORC1_SIGNALING"]

path_temp = '/data/kanferg/Sptial_Omics/projects/NatalieLab/liver_cancer/spatialomicstoolkit/temp/'
enrich_df = pd.read_csv(f"{path_temp}enrichmentmap_enrichments_version_3.tsv",sep='\t')
enrich_df = enrich_df.drop_duplicates(subset="Name", keep="first")
enrich_df["LeadingEdgeGenes"] = enrich_df["LeadingEdgeGenes"].str.split(",").apply(
    lambda x: [g.strip() for g in x] if isinstance(x, list) else x
)

enrich_df.index = enrich_df['Name']

from tqdm import tqdm
def campute_pathways(adata,table,pathway):
    genes = table.loc[pathway,'LeadingEdgeGenes']
    for gene in tqdm(genes):
        _ = vp.spatial.local_moran(adata, gene, graph_name=knn_graph)
    return adata.obsm['local_moran']

df_list = []
for pathway in pathways:
    print(pathway)
    df = campute_pathways(adata,enrich_df,pathway)
    df['in_lesion'] = adata.obs['in_lesion'].values
    df_moran_agg = df.groupby('in_lesion').mean().T
    df_moran_agg['pathway'] = pathway
    df_list.append(df_moran_agg)
    
df_pathway_moran = pd.concat(df_list, ignore_index=False)
df_pathway_moran['genes'] = df_pathway_moran.index

rnk = pd.read_csv(f"{path_temp}summary_DGE_Lesion_0vs1_version_3.csv")
rnk.rename(columns={'Unnamed: 0':'genes'}, inplace=True)

df_moran = pd.read_csv(f"{path_temp}lesion_vs_nonlesion_gene_localmoran_by_pathway_version_3.csv")

path_temp = '/data/kanferg/Sptial_Omics/projects/NatalieLab/liver_cancer/spatialomicstoolkit/temp/'
df_pathway_moran =  pd.read_csv(f"{path_temp}lesion_vs_nonlesion_gene_localmoran_by_pathway_version_3.csv")
rnk = pd.read_csv(f"{path_temp}summary_DGE_Lesion_0vs1_version_3.csv")
rnk.rename(columns={'Unnamed: 0':'genes'}, inplace=True)

df_moran['morans_I_FC'] = df_moran['True'] - df_moran['False']
merged_df = pd.merge(df_moran, rnk, on='genes', how='inner')

# ── Publication-quality scatter: Spatial autocorrelation vs. Lesion DE ────────
import matplotlib as mpl
import matplotlib.pyplot as plt
import scipy.stats as stats

# ── 1. Identify significance column ──────────────────────────────────────────
gene_col = "genes"
x_col    = "morans_I_FC"
y_col    = "log2FoldChange"

def _rank_sig(c):
    cl = c.lower()
    _has_pval = any(k in cl for k in (
        "fdr", "padj", "qval", "adj",
        "pval", "p_val", "p-val", "pvalue", "p_value", "sim_p"
    )) or cl.endswith("p") or cl.endswith("_p")
    if not _has_pval:
        return 0
    score = 0
    if any(k in cl for k in ("moran", "morans")):  score += 10
    if any(k in cl for k in ("fdr", "padj", "qval", "adj")):  score += 4
    if any(k in cl for k in ("pval", "p_val", "p-val", "pvalue", "p_value", "sim_p")):  score += 2
    if cl.endswith("p") or cl.endswith("_p"):  score += 1
    return score

_sig_candidates = sorted(
    [c for c in merged_df.columns if _rank_sig(c) > 0],
    key=_rank_sig, reverse=True
)
if not _sig_candidates:
    raise ValueError("No significance column found. Inspect merged_df.columns and set sig_col manually.")
sig_col = _sig_candidates[0]
print(f"Significance column: '{sig_col}'")

# ── 2. Prepare data ───────────────────────────────────────────────────────────
df = (merged_df[[gene_col, x_col, y_col, sig_col]]
      .copy()
      .drop_duplicates(subset=gene_col, keep="first")
      .dropna(subset=[x_col, y_col, sig_col])
      .reset_index(drop=True))
df = df.loc[:, ~df.columns.duplicated(keep="first")]

# Z-score of spatial autocorrelation; label genes with z > 1.64 (one-tailed p < 0.05)
df["zscore_x"] = stats.zscore(df[x_col])
Z_THRESH = 1.64
sig_mask  = df["zscore_x"] > Z_THRESH
label_df  = df[sig_mask].sort_values(x_col, ascending=False)
print(f"Total genes: {len(df)} | z > {Z_THRESH}: {sig_mask.sum()}")

# ── 3. rcParams — journal single-column (3.5 in, 8 pt base) ──────────────────
# Standard: Nature/Cell single column = 88 mm (3.46 in), min font 6 pt
mpl.rcParams.update({
    "font.family":        "sans-serif",
    "font.size":           8,
    "axes.labelsize":      9,
    "axes.titlesize":      9,
    "xtick.labelsize":     8,
    "ytick.labelsize":     8,
    "axes.linewidth":      0.75,
    "xtick.major.width":   0.6,
    "ytick.major.width":   0.6,
    "xtick.major.size":    3.0,
    "ytick.major.size":    3.0,
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "pdf.fonttype":        42,
    "svg.fonttype":        "none",
})

# ── 4. Plot — 3.5 × 3.5 in (single column, square) ───────────────────────────
fig, ax = plt.subplots(figsize=(3.5, 3.5), dpi=300)

# Non-significant background points
ax.scatter(
    df.loc[~sig_mask, x_col], df.loc[~sig_mask, y_col],
    s=10, color="#BBBBBB", alpha=0.55, edgecolors="none", zorder=2,
)
# Significant foreground points (darker, outlined)
ax.scatter(
    label_df[x_col], label_df[y_col],
    s=16, color="#444444", alpha=0.90,
    edgecolors="black", linewidth=0.35, zorder=3,
)

ax.set_xlabel("Spatial autocorrelation (Moran's I)", fontsize=9)
ax.set_ylabel("$\\log_2$ fold change (lesion / non-lesion)", fontsize=9)

# ── 5. Gene labels (only z-significant genes) ────────────────────────────────
try:
    from adjustText import adjust_text
    texts = [
        ax.text(row[x_col], row[y_col], row[gene_col], fontsize=7)
        for _, row in label_df.iterrows()
    ]
    adjust_text(
        texts, ax=ax,
        arrowprops=dict(arrowstyle="-", lw=0.35, color="black"),
        expand_points=(1.4, 1.6), expand_text=(1.2, 1.3),
    )
    print("adjustText used.")
except ImportError:
    print("adjustText not found – using annotate fallback.")
    _dirs = [
        ( 8,  8), ( 8, -8), (-8,  8), (-8, -8),
        (12,  0), (-12, 0), ( 0, 12), ( 0,-12),
        (10,  5), (10, -5), (-10,  5), (-10, -5),
        (14,  8), (14, -8), (-14,  8), (-14, -8),
    ]
    for k, (_, row) in enumerate(label_df.iterrows()):
        dx, dy = _dirs[k % len(_dirs)]
        ax.annotate(
            row[gene_col],
            xy=(row[x_col], row[y_col]),
            xytext=(dx, dy), textcoords="offset points",
            fontsize=7,
            ha="left" if dx >= 0 else "right",
            va="bottom" if dy >= 0 else "top",
            arrowprops=dict(arrowstyle="-", lw=0.35, color="black",
                            shrinkA=0, shrinkB=2),
        )

plt.tight_layout()

# ── 6. Save ───────────────────────────────────────────────────────────────────
out_pdf = f"{path_temp}moran_vs_lfc_scatter.pdf"
out_png = f"{path_temp}moran_vs_lfc_scatter.png"
fig.savefig(out_pdf, dpi=300, bbox_inches="tight")
fig.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved:\n  {out_pdf}\n  {out_png}")


# distanes analysis
# mamba activate spatialdata_sq_v1 
import scanpy as sc
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt
import squidpy as sq
from matplotlib.colors import ListedColormap
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D
import textwrap
from matplotlib.ticker import MaxNLocator
import seaborn as sns
import random
import pandas as pd
# import voyagerpy as vp
# import geopandas as gpd
# import libpysal as lps
from collections import OrderedDict
import scipy.sparse as sp
import pickle
from scipy.sparse import csr_matrix
import re


import numpy as np
from sklearn.neighbors import NearestNeighbors

'''
agg_adata_pairs = sc.read_h5ad('/data/HiTIF/data/spatialomics/liver_cancer/data/models/comp_analysis/cell_assign_pair_sample_from_2umbins_zone_report_21.h5ad')
agg_adata_pairs.obs['sample'] = agg_adata_pairs.obs['sample'].astype(str)
path = '/data/HiTIF/data/spatialomics/liver_cancer/data/lesion_analysis'
adata_lesion = sc.read_h5ad(os.path.join(path, 'report_23_supp_3_nichFinder_hep_v1.h5ad'))
adata_lesion.obs['niche_gene_only_knn12_k3'] = adata_lesion.obs['niche_gene_only_knn12_k3'].replace({'0':"niche_a",'1':'niche_b','2':'niche_c'})
'''
agg_adata_pairs.obs['cell_type_16um_hpe_lesion'] = agg_adata_pairs.obs['cell_type_16um_hpe'].astype(str)
agg_adata_pairs.obs.loc[adata_lesion.obs.index.values,'cell_type_16um_hpe_lesion'] = adata_lesion.obs['niche_gene_only_knn12_k3']
adata_hep1 = agg_adata_pairs[agg_adata_pairs.obs['cell_type_16um_hpe_lesion'].isin(['niche_a','niche_b','niche_c','Hep1'])].copy()
adata_hep6 = agg_adata_pairs[agg_adata_pairs.obs['cell_type_16um_hpe_lesion'].isin(['niche_a','niche_b','niche_c','Hep6'])].copy()

coords = adata_hep1.obsm["spatial"]

nn = NearestNeighbors(n_neighbors=2)
nn.fit(coords)
distances, indices = nn.kneighbors(coords)

# column 0 = self distance (0), column 1 = nearest other cell
nn_dist = distances[:, 1]

print("Min:", np.min(nn_dist))
print("Median:", np.median(nn_dist))
print("Mean:", np.mean(nn_dist))
print("95th percentile:", np.percentile(nn_dist, 95))

# --- settings ---
celltype_col = "cell_type_16um_hpe_lesion"
group_col = "sample"   # WT / KO
target = "Hep1"
# interval = np.arange(0, 41, 2)   # 0,2,4,...,40 um
interval = np.arange(0, 500, 10)

dfs = []

for g in ["exp_2", "exp_4","exp_6"]:
    ad = agg_adata_pairs[agg_adata_pairs.obs[group_col] == g].copy()
    ad.obs[celltype_col] = ad.obs[celltype_col].astype("category")

    occ, dist = sq.gr.co_occurrence(
        ad,
        cluster_key=celltype_col,
        interval=interval,
        copy=True,
        show_progress_bar=False
    )

    cats = list(ad.obs[celltype_col].cat.categories)
    i = cats.index(target)

    df = pd.DataFrame({
        "distance": np.tile(dist[1:], len(cats)),   # <- important fix
        "partner": np.repeat(cats, len(dist) - 1),  # <- important fix
        "ratio": occ[i].ravel(),
        "group": g
    })
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
df = df[df["partner"] != target]

df = df.loc[df['partner'].isin(['niche_a','niche_b','niche_c'])]

niche_a = "#0c7cec"    
niche_b =  "#01060C"
niche_c  = "#ef1111" 

samples = ["exp_2", "exp_4","exp_6"]

fig, axes = plt.subplots(
    1, 3,
    figsize=(10, 2.4),   # larger because font is 12
    sharex=True,
    sharey=True
)
axes = np.ravel(axes)

for ax,sample in zip(axes, samples):
    sns.lineplot(
        data=df[df["group"] == sample],
        x="distance",
        y="ratio",
        hue="partner",
        ax=ax,
        palette=[niche_a, niche_b, niche_c]
    )
    ax.set_title(sample)
    ax.set_xlabel("Distance (um)")
    ax.set_ylabel("Co-occurrence ratio")
    ax.legend(title="Partner", loc="upper right")
    
# notebook supp_report_11_v12_spatial_proximity_analysis.ipynb
